# Part 1: Supervised stopping-reason baseline

Predict the five stopping reasons using only the 157 training trials with visible labels. Each trial may have multiple reasons, so we fit one binary logistic regression per reason. The 1,014 unlabeled training trials are kept separately for later semi-supervised experiments.

We evaluate on the supplied 78-trial validation set. The main metric is **sample-wise F1**; we also report macro F1 and F1 for each reason. This experiment uses only the 150 provided reason embeddings.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(150)]

## Load the supplied training and validation sets

No new random split is needed. IDs identify trials and are excluded from the model inputs. The test data are reserved for later predictions.

In [2]:
def resolve_part1_input(directory, name):
    """Locate supplied CSV inputs while leaving experiment outputs under part1/."""
    supplied = {"part1_train.csv", "part1_val.csv", "part1_test.csv", "part1_sample_submission.csv", "pathway_catalog.csv"}
    nested = directory / "data" / name
    return nested if name in supplied and nested.is_file() else directory / name

# Support running from the repository root, AML-Midterm, or part1.
cwd = Path.cwd()
candidates = [cwd, cwd / "part1", cwd / "AML-Midterm" / "part1"]
DATA_DIR = next(
    (directory for directory in candidates
     if ((directory / "part1_train.csv").is_file() or (directory / "data" / "part1_train.csv").is_file())),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Run from the repository root, AML-Midterm, or part1.")

train_df = pd.read_csv(resolve_part1_input(DATA_DIR, "part1_train.csv"))
val_df = pd.read_csv(resolve_part1_input(DATA_DIR, "part1_val.csv"))

assert train_df["ID"].is_unique and val_df["ID"].is_unique
assert set(train_df["ID"]).isdisjoint(val_df["ID"])
assert train_df[REASONS].isin([0, 1, -100]).all().all()
assert val_df[REASONS].isin([0, 1]).all().all()
for frame in (train_df, val_df):
    assert np.isfinite(frame[EMBEDDING_COLUMNS].to_numpy()).all()

## Keep trials with visible labels

A reason label of `0` is a known negative and `1` is a known positive. A value of `-100` means the label is hidden; it must not become a negative training target.

In [3]:
labeled_mask = train_df[REASONS].isin([0, 1]).all(axis=1)
unlabeled_mask = train_df[REASONS].eq(-100).all(axis=1)
assert (labeled_mask | unlabeled_mask).all(), "Unexpected partially labeled row."

labeled_train_df = train_df.loc[labeled_mask].copy()
unlabeled_train_df = train_df.loc[unlabeled_mask].copy()

X_train = labeled_train_df[EMBEDDING_COLUMNS].to_numpy()
y_train = labeled_train_df[REASONS].to_numpy(dtype=int)
X_val = val_df[EMBEDDING_COLUMNS].to_numpy()
y_val = val_df[REASONS].to_numpy(dtype=int)

# Every binary classifier needs both positive and negative examples.
assert ((y_train.sum(axis=0) > 0) & (y_train.sum(axis=0) < len(y_train))).all()

display(pd.DataFrame({
    "dataset": ["Labeled training", "Unlabeled training (reserved)", "Validation"],
    "trials": [len(labeled_train_df), len(unlabeled_train_df), len(val_df)],
    "used to fit this baseline": [True, False, False],
}))

,dataset,trials,used to fit this baseline
0,Labeled training,157,True
1,Unlabeled training (reserved),1014,False
2,Validation,78,False


## Reusable classifier training and evaluation

Use the same training function for the supervised baseline and every augmented dataset. It fits a fresh scaler on the supplied training rows and five one-vs-rest L2 logistic regressions. Optional sample weights affect the classifiers; scaling remains unweighted. Scikit-learn metadata routing passes these weights to each binary estimator.

Evaluation always uses known labels, with a configurable prediction rule. Semi-supervised comparisons use the submitted baseline's 0.5 threshold and highest-probability fallback for empty predictions.

In [4]:
from sklearn import config_context


def train_reason_classifier(X, y, C=1000.0, sample_weight=None):
    """Fit training-only scaling and five logistic regressions."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=int)
    if X.ndim != 2 or y.shape != (len(X), len(REASONS)):
        raise ValueError("Expected one feature row and five binary labels per trial.")
    if not np.isfinite(X).all() or not np.isin(y, [0, 1]).all():
        raise ValueError("Features must be finite and training labels must be known.")
    if ((y.sum(axis=0) == 0) | (y.sum(axis=0) == len(y))).any():
        raise ValueError("Every reason needs both positive and negative examples.")
    if sample_weight is not None:
        sample_weight = np.asarray(sample_weight, dtype=float)
        if sample_weight.shape != (len(X),) or not np.isfinite(sample_weight).all():
            raise ValueError("Expected one finite sample weight per training trial.")
        if (sample_weight <= 0).any():
            raise ValueError("Pass included training rows with strictly positive weights.")

    with config_context(enable_metadata_routing=True):
        model = Pipeline([
            ("scaler", StandardScaler().set_fit_request(sample_weight=False)),
            ("classifier", OneVsRestClassifier(
                LogisticRegression(
                    C=C, solver="lbfgs", max_iter=5000, tol=1e-4,
                    random_state=SEED,
                ).set_fit_request(sample_weight=True)
            )),
        ])
        fit_parameters = {} if sample_weight is None else {"sample_weight": sample_weight}
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(X, y, **fit_parameters)
    return model


def predict_reason_labels(probabilities, threshold=0.5, ensure_nonempty=True):
    """Turn independent reason probabilities into a multi-label prediction."""
    probabilities = np.asarray(probabilities)
    if probabilities.ndim != 2 or probabilities.shape[1] != len(REASONS):
        raise ValueError("Expected five probabilities per trial.")
    if not np.isfinite(probabilities).all() or ((probabilities < 0) | (probabilities > 1)).any():
        raise ValueError("Probabilities must be finite and between zero and one.")
    predicted = (probabilities >= threshold).astype(int)
    empty = predicted.sum(axis=1) == 0
    if ensure_nonempty:
        rows = np.flatnonzero(empty)
        predicted[rows, probabilities[rows].argmax(axis=1)] = 1
    return predicted, empty


def evaluate_reason_probabilities(probabilities, y, threshold=0.5, ensure_nonempty=True):
    """Return sample/macro/per-reason F1 plus auditable probabilities and labels."""
    actual = np.asarray(y, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    if actual.shape != probabilities.shape or not np.isin(actual, [0, 1]).all():
        raise ValueError("Evaluation requires five known binary labels per trial.")
    predictions, empty = predict_reason_labels(probabilities, threshold, ensure_nonempty)
    metrics = {
        "sample_f1": f1_score(actual, predictions, average="samples", zero_division=0),
        "macro_f1": f1_score(actual, predictions, average="macro", zero_division=0),
        "fallback_trials": int(empty.sum()) if ensure_nonempty else 0,
        **dict(zip(
            [f"f1_{reason}" for reason in REASONS],
            f1_score(actual, predictions, average=None, zero_division=0),
        )),
    }
    return {"metrics": metrics, "probabilities": probabilities,
            "predictions": predictions, "fallback": empty if ensure_nonempty else np.zeros(len(y), dtype=bool)}


def evaluate_reason_classifier(model, X, y, threshold=0.5, ensure_nonempty=True):
    """Predict with a fitted classifier and apply the shared evaluation rules."""
    return evaluate_reason_probabilities(
        model.predict_proba(X), y, threshold=threshold, ensure_nonempty=ensure_nonempty
    )

## Standardize and fit logistic regression

The pipeline learns one mean and standard deviation per embedding column from the **157 labeled training trials only**. It reuses those statistics when predicting validation trials.

Each of the five classifiers uses L2-regularized logistic regression with `C=1`. We fix the probability threshold at 0.5 and do not tune settings on validation for this initial baseline. A trial can receive several reasons or none; no single-reason constraint is imposed.

In [5]:
baseline_model = train_reason_classifier(X_train, y_train, C=1.0)
assert baseline_model.named_steps["scaler"].n_samples_seen_ == len(y_train)
print(f"Fitted five classifiers on {len(y_train)} labeled training trials.")

Fitted five classifiers on 157 labeled training trials.


## Evaluate on the same validation set used for future comparisons

Sample-wise F1 computes F1 for each trial's set of reasons and averages across trials. Macro F1 averages the five per-reason F1 scores. Training scores help reveal overfitting; validation scores measure performance on held-out trials. Undefined F1 values are assigned zero.

In [6]:
THRESHOLD = 0.5
train_probabilities = baseline_model.predict_proba(X_train)
val_probabilities = baseline_model.predict_proba(X_val)
train_predictions = (train_probabilities >= THRESHOLD).astype(int)
val_predictions = (val_probabilities >= THRESHOLD).astype(int)

for probabilities, labels in ((train_probabilities, y_train), (val_probabilities, y_val)):
    assert probabilities.shape == labels.shape
    assert np.isfinite(probabilities).all()
    assert ((probabilities >= 0) & (probabilities <= 1)).all()

score_rows = []
for split, actual, predicted in (
    ("Training", y_train, train_predictions),
    ("Validation", y_val, val_predictions),
):
    score_rows.append({
        "split": split,
        "sample-wise F1": f1_score(actual, predicted, average="samples", zero_division=0),
        "macro F1": f1_score(actual, predicted, average="macro", zero_division=0),
    })
baseline_scores = pd.DataFrame(score_rows).set_index("split")
display(baseline_scores.round(4))

per_reason_scores = pd.DataFrame({
    "reason": REASONS,
    "training positives": y_train.sum(axis=0),
    "validation positives": y_val.sum(axis=0),
    "validation predicted positives": val_predictions.sum(axis=0),
    "validation F1": f1_score(y_val, val_predictions, average=None, zero_division=0),
}).set_index("reason")
display(per_reason_scores.round(4))

print(f"Validation trials with no predicted reason: {(val_predictions.sum(axis=1) == 0).sum()}")
print(f"Validation sample-wise F1: {baseline_scores.loc['Validation', 'sample-wise F1']:.4f}")

,sample-wise F1,macro F1
split,,
Training,1.0000,1.0000
Validation,0.8162,0.6685


,training positives,validation positives,validation predicted positives,validation F1
reason,,,,
Safety,16,9,6,0.8000
Funding,8,4,1,0.4000
Efficacy,17,14,8,0.6364
Recruitment,101,50,50,0.9800
Other,25,10,9,0.5263


Validation trials with no predicted reason: 8
Validation sample-wise F1: 0.8162


## Compare with predicting Recruitment for every trial

This simple reference helps show whether the model adds value beyond the most common stopping reason. Both methods are evaluated on the identical validation trials.

In [7]:
recruitment_predictions = np.zeros_like(y_val)
recruitment_predictions[:, REASONS.index("Recruitment")] = 1

reference_comparison = pd.DataFrame([
    {
        "method": "Always Recruitment",
        "validation sample-wise F1": f1_score(
            y_val, recruitment_predictions, average="samples", zero_division=0
        ),
        "validation macro F1": f1_score(
            y_val, recruitment_predictions, average="macro", zero_division=0
        ),
    },
    {
        "method": "Labeled-only logistic regression",
        "validation sample-wise F1": baseline_scores.loc["Validation", "sample-wise F1"],
        "validation macro F1": baseline_scores.loc["Validation", "macro F1"],
    },
]).set_index("method")
display(reference_comparison.round(4))

,validation sample-wise F1,validation macro F1
method,,
Always Recruitment,0.6239,0.1562
Labeled-only logistic regression,0.8162,0.6685


## Baseline results

On the 78 validation trials, labeled-only logistic regression achieves **sample-wise F1 = 0.8162** and **macro F1 = 0.6685**. Always predicting Recruitment achieves sample-wise F1 = 0.6239, so logistic regression improves this reference by 0.1923 (19.23 percentage points).

Per-reason validation F1 is 0.8000 for Safety, 0.4000 for Funding, 0.6364 for Efficacy, 0.9800 for Recruitment, and 0.5263 for Other. Funding has only four validation positives, so a few changed predictions can substantially change its score. Eight validation trials receive no predicted reason at the fixed 0.5 threshold.

Training F1 is 1.0000 for both sample-wise and macro averages. The model fits the small labeled training set perfectly but makes held-out errors, so training scores alone cannot guide model selection. These results are a fixed, untuned validation baseline, not a test or Kaggle score.

## Compare L2 regularization strengths

Compare the prespecified values `C = 0.01, 0.1, 1, 10`. Smaller `C` means stronger regularization. Keep the same 157 labeled training trials, 78 validation trials, embedding features, training-only standardization, and 0.5 probability threshold.

The original `C=1` baseline remains unchanged. Choose the supervised comparison model by validation sample-wise F1; exact ties favor smaller `C`. Also inspect macro F1 and each reason's F1. Because this step uses validation performance to select `C`, its winning score is a tuning result rather than an independent test estimate.

In [8]:
from sklearn.base import clone

C_VALUES = [0.01, 0.1, 1.0, 10.0]
regularization_models = {}
regularization_val_predictions = {}
regularization_rows = []
regularization_reason_rows = []

for C in C_VALUES:
    if C == 1.0:
        # Reuse the fitted original baseline without changing it.
        model = baseline_model
    else:
        model = clone(baseline_model).set_params(classifier__estimator__C=C)
        with warnings.catch_warnings():
            warnings.simplefilter("error", ConvergenceWarning)
            model.fit(X_train, y_train)

    assert model.named_steps["scaler"].n_samples_seen_ == len(y_train)
    train_pred = (model.predict_proba(X_train) >= THRESHOLD).astype(int)
    val_pred = (model.predict_proba(X_val) >= THRESHOLD).astype(int)
    regularization_models[C] = model
    regularization_val_predictions[C] = val_pred
    regularization_rows.append({
        "C": C,
        "training sample-wise F1": f1_score(
            y_train, train_pred, average="samples", zero_division=0
        ),
        "validation sample-wise F1": f1_score(
            y_val, val_pred, average="samples", zero_division=0
        ),
        "validation macro F1": f1_score(
            y_val, val_pred, average="macro", zero_division=0
        ),
        "validation trials with no predicted reason": int((val_pred.sum(axis=1) == 0).sum()),
    })
    regularization_reason_rows.append({
        "C": C,
        **dict(zip(REASONS, f1_score(y_val, val_pred, average=None, zero_division=0))),
    })

regularization_results = pd.DataFrame(regularization_rows).set_index("C")
regularization_reason_f1 = pd.DataFrame(regularization_reason_rows).set_index("C")
print("Overall F1 comparison:")
display(regularization_results.round(4))
print("Per-reason validation F1:")
display(regularization_reason_f1.round(4))

selected_C = float(
    regularization_results.reset_index().sort_values(
        ["validation sample-wise F1", "C"],
        ascending=[False, True],
        kind="stable",
    ).iloc[0]["C"]
)
selected_supervised_model = regularization_models[selected_C]
selected_validation_f1 = regularization_results.loc[selected_C, "validation sample-wise F1"]
original_validation_f1 = baseline_scores.loc["Validation", "sample-wise F1"]
print(f"Selected C: {selected_C:g}")
print(f"Selected validation sample-wise F1: {selected_validation_f1:.4f}")
print(f"Change from original C=1 baseline: {selected_validation_f1 - original_validation_f1:+.4f}")

Overall F1 comparison:


,training sample-wise F1,validation sample-wise F1,validation macro F1,validation trials with no predicted reason
C,,,,
0.01,0.8705,0.6880,0.3641,22
0.10,0.9979,0.7991,0.5829,13
1.00,1.0000,0.8162,0.6685,8
10.00,1.0000,0.8248,0.6804,8


Per-reason validation F1:


,Safety,Funding,Efficacy,Recruitment,Other
C,,,,,
0.01,0.3636,0.0,0.1333,0.9901,0.3333
0.10,0.7143,0.0,0.6667,1.0000,0.5333
1.00,0.8000,0.4,0.6364,0.9800,0.5263
10.00,0.8000,0.4,0.6957,0.9800,0.5263


Selected C: 10
Selected validation sample-wise F1: 0.8248
Change from original C=1 baseline: +0.0085


## Regularization comparison results

Among the four prespecified settings, `C=10` has the highest validation sample-wise F1: **0.8248**, compared with **0.8162** for the original `C=1` baseline. Validation macro F1 increases from 0.6685 to 0.6804. The sample-wise improvement is 0.0085, or approximately 0.85 percentage points. With only 78 validation trials, this small gain should not be treated as proof of better test performance.

Stronger regularization did not improve this comparison: `C=0.1` achieves sample-wise F1 of 0.7991 and `C=0.01` achieves 0.6880. Both fail to predict any Funding positives at the fixed threshold. At `C=0.01`, the model predicts no reason for 22 validation trials, compared with eight for `C=1` and `C=10`; this is consistent with overly restrictive weights suppressing positive predictions, particularly for rare reasons.

The selected `C=10` model uses weaker regularization. Its training sample-wise F1 remains 1.0000, while Efficacy validation F1 improves from 0.6364 to 0.6957 and the other four per-reason F1 scores remain unchanged. Perfect training F1 alone therefore did not establish that stronger regularization would improve generalization. Keep the original baseline and the tuned supervised result for subsequent comparisons.

## One follow-up check: C=100

After the initial four-value comparison selected `C=10` at the upper edge of the grid, test `C=100` once. This follow-up was chosen after inspecting the earlier results; it was not part of the original prespecified grid. Keep the data, preprocessing, threshold, and other model settings fixed. Compare with `C=10`, and select by sample-wise validation F1 with ties favoring smaller `C`.

In [9]:
followup_C = 100.0
followup_model = clone(baseline_model).set_params(classifier__estimator__C=followup_C)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    followup_model.fit(X_train, y_train)

assert followup_model.named_steps["scaler"].n_samples_seen_ == len(y_train)
followup_train_predictions = (followup_model.predict_proba(X_train) >= THRESHOLD).astype(int)
followup_val_predictions = (followup_model.predict_proba(X_val) >= THRESHOLD).astype(int)
regularization_models[followup_C] = followup_model
regularization_val_predictions[followup_C] = followup_val_predictions
regularization_results.loc[followup_C] = {
    "training sample-wise F1": f1_score(
        y_train, followup_train_predictions, average="samples", zero_division=0
    ),
    "validation sample-wise F1": f1_score(
        y_val, followup_val_predictions, average="samples", zero_division=0
    ),
    "validation macro F1": f1_score(
        y_val, followup_val_predictions, average="macro", zero_division=0
    ),
    "validation trials with no predicted reason": int((followup_val_predictions.sum(axis=1) == 0).sum()),
}
regularization_reason_f1.loc[followup_C] = f1_score(
    y_val, followup_val_predictions, average=None, zero_division=0
)

print("C=10 versus C=100:")
display(regularization_results.loc[[10.0, followup_C]].round(4))
print("Per-reason validation F1:")
display(regularization_reason_f1.loc[[10.0, followup_C]].round(4))

selected_C = float(
    regularization_results.reset_index().sort_values(
        ["validation sample-wise F1", "C"],
        ascending=[False, True],
        kind="stable",
    ).iloc[0]["C"]
)
selected_supervised_model = regularization_models[selected_C]
selected_validation_f1 = regularization_results.loc[selected_C, "validation sample-wise F1"]
followup_change = (
    regularization_results.loc[followup_C, "validation sample-wise F1"]
    - regularization_results.loc[10.0, "validation sample-wise F1"]
)
print(f"Change from C=10 to C=100: {followup_change:+.4f}")
print(f"Validation trials with any changed label prediction: {(followup_val_predictions != regularization_val_predictions[10.0]).any(axis=1).sum()}")
print(f"Selected C after the follow-up: {selected_C:g}")
print(f"Selected validation sample-wise F1: {selected_validation_f1:.4f}")

C=10 versus C=100:


,training sample-wise F1,validation sample-wise F1,validation macro F1,validation trials with no predicted reason
C,,,,
10.0,1.0,0.8248,0.6804,8
100.0,1.0,0.8462,0.7677,8


Per-reason validation F1:


,Safety,Funding,Efficacy,Recruitment,Other
C,,,,,
10.0,0.8000,0.4000,0.6957,0.9800,0.5263
100.0,0.9412,0.6667,0.6957,0.9796,0.5556


Change from C=10 to C=100: +0.0214
Validation trials with any changed label prediction: 8
Selected C after the follow-up: 100
Selected validation sample-wise F1: 0.8462


## Follow-up result

`C=100` improves validation sample-wise F1 from **0.8248 to 0.8462** (approximately 2.14 percentage points) and macro F1 from **0.6804 to 0.7677**. Training sample-wise F1 remains 1.0000. Eight validation trials have at least one changed predicted label relative to `C=10`.

Safety F1 increases from 0.8000 to 0.9412, Funding from 0.4000 to 0.6667, and Other from 0.5263 to 0.5556. Efficacy remains 0.6957, while Recruitment decreases slightly from 0.9800 to 0.9796. Both models predict no reason for eight validation trials.

After this check, `C=100` is the selected supervised comparator. The final `C=1000` check is recorded separately below. Keep the original `C=1` baseline for reference. This selection used the same small validation set, so the gain is a validation tuning result rather than an independently confirmed test improvement.

## Final regularization check: C=1000

After the improvement at `C=100`, test one final weaker regularization setting, `C=1000`. This is a second adaptive follow-up to the original four-value grid. Use the same 157 labeled training trials, 78 validation trials, training-only standardization, and fixed 0.5 threshold. Select by validation sample-wise F1; ties favor smaller `C`. Stop this search after this check.

In [10]:
final_check_C = 1000.0
final_check_model = clone(baseline_model).set_params(classifier__estimator__C=final_check_C)
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    final_check_model.fit(X_train, y_train)

assert final_check_model.named_steps["scaler"].n_samples_seen_ == len(y_train)
final_check_train_predictions = (final_check_model.predict_proba(X_train) >= THRESHOLD).astype(int)
final_check_val_predictions = (final_check_model.predict_proba(X_val) >= THRESHOLD).astype(int)
regularization_models[final_check_C] = final_check_model
regularization_val_predictions[final_check_C] = final_check_val_predictions
regularization_results.loc[final_check_C] = {
    "training sample-wise F1": f1_score(
        y_train, final_check_train_predictions, average="samples", zero_division=0
    ),
    "validation sample-wise F1": f1_score(
        y_val, final_check_val_predictions, average="samples", zero_division=0
    ),
    "validation macro F1": f1_score(
        y_val, final_check_val_predictions, average="macro", zero_division=0
    ),
    "validation trials with no predicted reason": int((final_check_val_predictions.sum(axis=1) == 0).sum()),
}
regularization_reason_f1.loc[final_check_C] = f1_score(
    y_val, final_check_val_predictions, average=None, zero_division=0
)

print("C=100 versus C=1000:")
display(regularization_results.loc[[100.0, final_check_C]].round(4))
print("Per-reason validation F1:")
display(regularization_reason_f1.loc[[100.0, final_check_C]].round(4))

selected_C = float(
    regularization_results.reset_index().sort_values(
        ["validation sample-wise F1", "C"],
        ascending=[False, True],
        kind="stable",
    ).iloc[0]["C"]
)
selected_supervised_model = regularization_models[selected_C]
selected_validation_f1 = regularization_results.loc[selected_C, "validation sample-wise F1"]
final_check_change = (
    regularization_results.loc[final_check_C, "validation sample-wise F1"]
    - regularization_results.loc[100.0, "validation sample-wise F1"]
)
print(f"Change from C=100 to C=1000: {final_check_change:+.4f}")
print(f"Validation trials with any changed label prediction: {(final_check_val_predictions != regularization_val_predictions[100.0]).any(axis=1).sum()}")
print(f"Selected C after the final check: {selected_C:g}")
print(f"Selected validation sample-wise F1: {selected_validation_f1:.4f}")

C=100 versus C=1000:


,training sample-wise F1,validation sample-wise F1,validation macro F1,validation trials with no predicted reason
C,,,,
100.0,1.0,0.8462,0.7677,8
1000.0,1.0,0.8803,0.8133,4


Per-reason validation F1:


,Safety,Funding,Efficacy,Recruitment,Other
C,,,,,
100.0,0.9412,0.6667,0.6957,0.9796,0.5556
1000.0,0.9412,0.6667,0.8571,0.9697,0.6316


Change from C=100 to C=1000: +0.0342
Validation trials with any changed label prediction: 7
Selected C after the final check: 1000
Selected validation sample-wise F1: 0.8803


## Final regularization result

`C=1000` improves validation sample-wise F1 from **0.8462 to 0.8803** (approximately 3.42 percentage points) and macro F1 from **0.7677 to 0.8133**. Training sample-wise F1 remains 1.0000. Seven validation trials have at least one changed predicted label relative to `C=100`; the number of trials with no predicted reason falls from eight to four.

Efficacy validation F1 increases from 0.6957 to 0.8571 and Other from 0.5556 to 0.6316. Safety stays at 0.9412 and Funding at 0.6667, while Recruitment decreases from 0.9796 to 0.9697.

Select **`C=1000`** as the tuned supervised comparator and end this regularization search. Preserve the original `C=1` baseline (sample-wise F1 0.8162). All settings used the same labeled training trials, training-only standardization, and fixed 0.5 prediction threshold. The selected score used the same 78 validation trials for tuning, so it does not establish performance on an independent test set.

## Next comparison

Keep both the original `C=1` baseline and the model selected in the regularization comparison and its two follow-ups (`C=100` and `C=1000`) as supervised references when evaluating semi-supervised methods that use the 1,014 unlabeled training trials. Use the same validation trials and metrics, and report the tuning performed for each method.

The experiments so far use only reason embeddings and a fixed 0.5 probability threshold. Additional features and generated labels remain later steps. The supervised test submission is exported below, with an explicitly documented rule for empty prediction sets. Rare categories have few validation examples, so small changes need cautious interpretation.

## Export the supervised Kaggle baseline

Use the fitted **`C=1000`** supervised pipeline, trained on the original 157 labeled training trials. Reuse its training means and standard deviations for the test embeddings. Do not retrain on validation or unlabeled trials.

Keep the 0.5 prediction threshold. For submission, if no reason exceeds the threshold, choose the highest-probability reason instead: every Part 1 trial has at least one stopping reason, and this ensures a nonempty reason string in every CSV row. This is an explicit postprocessing change from the earlier threshold-only baseline. Report its validation score separately so the Kaggle baseline and later comparisons use the same prediction rule.

Write one row per test trial with columns `ID` and `Predicted`, using space-separated reason names in the sample submission's ID order. This cell creates the CSV locally; upload it manually to the Part 1 competition.

In [11]:
def submission_reason_predictions(probabilities):
    return predict_reason_labels(probabilities, threshold=THRESHOLD, ensure_nonempty=True)


submission_model = regularization_models[1000.0]
assert submission_model.named_steps["classifier"].estimator.C == 1000.0
assert submission_model.named_steps["scaler"].n_samples_seen_ == len(y_train)

# Score the exact submission rule on validation before exporting test labels.
submission_val_probabilities = submission_model.predict_proba(X_val)
submission_val_predictions, val_needs_reason = submission_reason_predictions(
    submission_val_probabilities
)
submission_validation_f1 = f1_score(
    y_val, submission_val_predictions, average="samples", zero_division=0
)
submission_validation_macro_f1 = f1_score(
    y_val, submission_val_predictions, average="macro", zero_division=0
)
display(pd.DataFrame([
    {
        "prediction rule": "C=1000, threshold only",
        "validation sample-wise F1": regularization_results.loc[1000.0, "validation sample-wise F1"],
        "validation macro F1": regularization_results.loc[1000.0, "validation macro F1"],
    },
    {
        "prediction rule": "C=1000, threshold plus nonempty fallback (submitted)",
        "validation sample-wise F1": submission_validation_f1,
        "validation macro F1": submission_validation_macro_f1,
    },
]).set_index("prediction rule").round(4))
print(f"Validation trials requiring fallback: {val_needs_reason.sum()}")

test_df = pd.read_csv(resolve_part1_input(DATA_DIR, "part1_test.csv"))
sample_submission = pd.read_csv(resolve_part1_input(DATA_DIR, "part1_sample_submission.csv"))
assert test_df["ID"].is_unique and sample_submission["ID"].is_unique
assert set(test_df["ID"]) == set(sample_submission["ID"])
assert set(test_df["ID"]).isdisjoint(train_df["ID"])
assert set(test_df["ID"]).isdisjoint(val_df["ID"])
assert np.isfinite(test_df[EMBEDDING_COLUMNS].to_numpy()).all()

test_probabilities = submission_model.predict_proba(test_df[EMBEDDING_COLUMNS].to_numpy())
assert test_probabilities.shape == (len(test_df), len(REASONS))
assert np.isfinite(test_probabilities).all()
assert ((test_probabilities >= 0) & (test_probabilities <= 1)).all()
test_predictions, test_needs_reason = submission_reason_predictions(test_probabilities)

reason_strings = [
    " ".join(reason for reason, present in zip(REASONS, row) if present)
    for row in test_predictions
]
submission = pd.DataFrame({"ID": test_df["ID"], "Predicted": reason_strings})
submission = submission.set_index("ID").loc[sample_submission["ID"]].reset_index()
assert submission["ID"].tolist() == sample_submission["ID"].tolist()
assert submission.columns.tolist() == ["ID", "Predicted"]
assert submission["Predicted"].str.len().gt(0).all()
assert submission["Predicted"].map(
    lambda value: set(value.split()).issubset(REASONS)
).all()

submission_path = DATA_DIR / "submissions/archive/submitted/logistic_baseline__kaggle_0.87072.csv"
submission.to_csv(submission_path, index=False)
saved_submission = pd.read_csv(submission_path)
pd.testing.assert_frame_equal(saved_submission, submission)
print(f"Saved {len(submission)} test predictions to: {submission_path.resolve()}")
print(f"Test trials requiring fallback: {test_needs_reason.sum()}")
display(submission.head(10))
display(pd.Series(test_predictions.sum(axis=0), index=REASONS, name="test predicted positives"))

,validation sample-wise F1,validation macro F1
prediction rule,,
"C=1000, threshold only",0.8803,0.8133
"C=1000, threshold plus nonempty fallback (submitted)",0.8932,0.8043


Validation trials requiring fallback: 4
Saved 312 test predictions to: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/submissions/archive/submitted/logistic_baseline__kaggle_0.87072.csv
Test trials requiring fallback: 16


,ID,Predicted
0,TR00756,Efficacy
1,TR05267,Safety Other
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Efficacy
5,TR01460,Recruitment
6,TR00199,Funding Recruitment
7,TR09959,Recruitment
8,TR01886,Recruitment
9,TR09553,Efficacy Other


Safety          58
Funding          7
Efficacy        57
Recruitment    179
Other           63
Name: test predicted positives, dtype: int64

## Recorded supervised baseline for future comparisons

This is the supervised baseline submitted to Kaggle. All regularization experiments, validation outputs, and the reproducible CSV export are retained above.

| Setting | Recorded value |
|---|---|
| Features | `emb_0` through `emb_149` only |
| Training data | 157 trials with complete reason labels |
| Unlabeled data | 1,014 trials reserved; unused in this model |
| Validation data | The supplied 78 labeled trials |
| Preprocessing | StandardScaler fitted on the 157 labeled training trials |
| Classifier | Five one-vs-rest L2 logistic regressions |
| Selected regularization | `C=1000` |
| Other settings | `solver="lbfgs"`, `max_iter=5000`, `tol=1e-4`, `random_state=42` |
| Selection history | Initial `C=0.01, 0.1, 1, 10`; adaptive follow-ups `100` and `1000` |
| Prediction rule | Select every probability >= 0.5; if none qualify, select the highest-probability reason |
| Fallback cases | 4 validation trials; 16 test trials |
| Validation sample-wise F1, submitted rule | **0.8932** |
| Validation macro F1, submitted rule | **0.8043** |
| Validation sample-wise / macro F1, threshold-only rule | 0.8803 / 0.8133 |
| Training sample-wise / macro F1 | 1.0000 / 1.0000 |
| Kaggle leaderboard score | **0.87072** |
| Submission file | `submissions/archive/submitted/logistic_baseline__kaggle_0.87072.csv` |
| Test predictions | 312 rows, columns `ID` and `Predicted` |

**Kaggle provenance:** Michael reported the score after manually uploading this submission. The leaderboard snapshot was rank **32**, with the leading score **0.92735**. These values were reported in the conversation, not independently retrieved; rank and leading score can change. The validation-to-Kaggle sample-wise F1 difference is approximately 0.0224 (2.24 percentage points, using the unrounded validation score).

**Comparison protocol:** evaluate future semi-supervised methods on the same 78 validation trials, using the same nonempty-label prediction rule, sample-wise F1, macro F1, and per-reason F1. Compare their Kaggle scores with **0.87072** and document any tuning or changes in training data. The supervised model was not retrained on validation for this submission. Validation results were used for model selection.

Submission SHA-256: `bbd05ba89d4ac0a21864bbdd029a21633f2ee3ead391f25eda3148d73a276c10`.


# K-means label augmentation

Compare **K = 2, 5, 8, 10, 15**, keeping 150 embedding dimensions and using no additional PCA. K-means is fitted only on the 1,171 training trials, with a separate StandardScaler fitted on those same training embeddings. Validation and test trials do not participate in clustering, pseudo-label generation, or classifier fitting.

For each cluster, estimate five reason prevalences using only its originally labeled trials. Smooth each prevalence toward the overall labeled-training prevalence with prior strength 2. A trial receives a complete generated label vector only when its cluster contains at least 3 labeled examples, every label's assigned-class confidence is at least 0.7, and at least one reason is positive. Label 1 corresponds to prevalence >= 0.5; otherwise label 0. These confidences are heuristics from cluster frequencies, not calibrated probabilities of correctness.

Known labels are preserved with weight 1. Generated labels receive weight 0.25. The confidence threshold, smoothing, support requirement, and weights are fixed across all K values; only K is selected using validation. The final classifier uses C=1000 and the same training function and prediction rule as the supervised reference. Its scaler is fitted afresh on each augmented training dataset.

Each experiment saves the exact training CSV, a separate audit of all 1,171 original training rows (including excluded rows), cluster statistics, validation predictions, and fitted models. Unknown labels remain -100 in excluded audit rows. All original input CSVs and the submitted baseline file are preserved.

In [12]:
import hashlib
import json
import platform
import joblib
import sklearn
from sklearn.cluster import KMeans
from threadpoolctl import threadpool_limits

K_VALUES = [2, 5, 8, 10, 15]
# Scores reported by Michael after manual submissions; no automated Kaggle access.
REPORTED_KAGGLE_SCORES = {"baseline_C1000": 0.87072, "k02": 0.87393}
KMEANS_SETTINGS = {
    "random_state": SEED, "n_init": 10, "max_iter": 300, "tol": 1e-4,
    "init": "k-means++", "algorithm": "lloyd",
}
LABEL_SETTINGS = {
    "min_labeled_support": 3,
    "prior_strength": 2.0,
    "min_confidence": 0.7,
    "pseudo_weight": 0.25,
}
EXPERIMENT_DIR = (DATA_DIR / "experiments" / "kmeans").resolve()
for name in ["datasets", "label_audits", "clusters", "models", "validation_predictions"]:
    (EXPERIMENT_DIR / name).mkdir(parents=True, exist_ok=True)

# The reusable function must reproduce the existing submitted baseline.
refactored_baseline = train_reason_classifier(X_train, y_train, C=1000.0)
baseline_evaluation = evaluate_reason_classifier(refactored_baseline, X_val, y_val)
np.testing.assert_allclose(
    baseline_evaluation["probabilities"], submission_val_probabilities, rtol=1e-10, atol=1e-12
)
np.testing.assert_array_equal(baseline_evaluation["predictions"], submission_val_predictions)
assert baseline_evaluation["metrics"]["sample_f1"] == submission_validation_f1

all_training_embeddings = train_df[EMBEDDING_COLUMNS].to_numpy()
kmeans_scaler = StandardScaler().fit(all_training_embeddings)
all_training_scaled = kmeans_scaler.transform(all_training_embeddings)
assert kmeans_scaler.n_samples_seen_ == len(train_df) == 1171
original_label_mask = train_df[REASONS].isin([0, 1]).all(axis=1).to_numpy()
assert original_label_mask.sum() == 157
print("Baseline reproduced:", baseline_evaluation["metrics"])
print("K values:", K_VALUES)
print("Fixed label-generation settings:", LABEL_SETTINGS)

Baseline reproduced: {'sample_f1': 0.893162393162393, 'macro_f1': 0.8043238108640585, 'fallback_trials': 4, 'f1_Safety': np.float64(0.9411764705882353), 'f1_Funding': np.float64(0.6666666666666666), 'f1_Efficacy': np.float64(0.8125), 'f1_Recruitment': np.float64(0.9696969696969697), 'f1_Other': np.float64(0.631578947368421)}
K values: [2, 5, 8, 10, 15]
Fixed label-generation settings: {'min_labeled_support': 3, 'prior_strength': 2.0, 'min_confidence': 0.7, 'pseudo_weight': 0.25}


In [13]:
def make_augmented_training_data(frame, cluster_ids, k):
    """Generate confident labels from known labels; preserve every original label."""
    known = frame[REASONS].isin([0, 1]).all(axis=1).to_numpy()
    original_labels = frame[REASONS].to_numpy(dtype=int)
    priors = original_labels[known].mean(axis=0)
    support = np.bincount(cluster_ids[known], minlength=k)
    sizes = np.bincount(cluster_ids, minlength=k)
    positives = np.vstack([
        original_labels[known & (cluster_ids == cluster)].sum(axis=0)
        for cluster in range(k)
    ])
    strength = LABEL_SETTINGS["prior_strength"]
    cluster_probabilities = (positives + strength * priors) / (support[:, None] + strength)
    cluster_labels = (cluster_probabilities >= 0.5).astype(int)
    assigned_confidence = np.where(cluster_labels == 1, cluster_probabilities, 1 - cluster_probabilities)
    cluster_confidence = assigned_confidence.min(axis=1)
    eligible_cluster = (
        (support >= LABEL_SETTINGS["min_labeled_support"])
        & (cluster_confidence >= LABEL_SETTINGS["min_confidence"])
        & (cluster_labels.sum(axis=1) > 0)
    )
    generated = ~known & eligible_cluster[cluster_ids]
    included = known | generated

    augmented = frame.copy()
    for i, reason in enumerate(REASONS):
        augmented[f"original_{reason}"] = original_labels[:, i]
        augmented[f"cluster_probability_{reason}"] = cluster_probabilities[cluster_ids, i]
    augmented.loc[generated, REASONS] = cluster_labels[cluster_ids[generated]]
    augmented["label_source"] = np.where(known, "original", np.where(generated, "generated", "unknown"))
    augmented["kmeans_cluster"] = cluster_ids
    augmented["cluster_labeled_support"] = support[cluster_ids]
    augmented["cluster_label_confidence"] = cluster_confidence[cluster_ids]
    augmented["included_in_training"] = included
    augmented["training_weight"] = np.where(known, 1.0, np.where(generated, LABEL_SETTINGS["pseudo_weight"], 0.0))
    training = augmented.loc[included].copy()
    audit_columns = ["ID"] + REASONS + [f"original_{r}" for r in REASONS] + [
        "label_source", "kmeans_cluster", "cluster_labeled_support", "cluster_label_confidence",
        "included_in_training", "training_weight",
    ] + [f"cluster_probability_{r}" for r in REASONS]
    audit = augmented[audit_columns].copy()
    clusters = pd.DataFrame({
        "cluster": np.arange(k), "trials": sizes, "labeled_trials": support,
        "unlabeled_trials": sizes - support, "min_assigned_confidence": cluster_confidence,
        "eligible_for_generated_labels": eligible_cluster,
    })
    for i, reason in enumerate(REASONS):
        clusters[f"known_positive_{reason}"] = positives[:, i]
        clusters[f"probability_{reason}"] = cluster_probabilities[:, i]

    np.testing.assert_array_equal(augmented.loc[known, REASONS].to_numpy(), original_labels[known])
    assert augmented.loc[~included, REASONS].eq(-100).all().all()
    assert training[REASONS].isin([0, 1]).all().all()
    assert set(training["ID"]).issubset(frame["ID"])
    assert set(training["ID"]).isdisjoint(val_df["ID"])
    assert len(training) == int(known.sum() + generated.sum())
    return training, audit, clusters


def save_validation_predictions(evaluation, path):
    output = val_df[["ID"] + REASONS].rename(columns={r: f"actual_{r}" for r in REASONS}).copy()
    for i, reason in enumerate(REASONS):
        output[f"probability_{reason}"] = evaluation["probabilities"][:, i]
        output[f"predicted_{reason}"] = evaluation["predictions"][:, i]
    output["used_fallback"] = evaluation["fallback"]
    output.to_csv(path, index=False)


def summarize_experiment(name, method, k, training, model, evaluation, dataset_path):
    original_training = evaluate_reason_classifier(model, X_train, y_train)
    return {
        "experiment": name, "method": method, "k": k, "C": 1000.0,
        "original_trials": int(training["label_source"].eq("original").sum()),
        "generated_trials": int(training["label_source"].eq("generated").sum()),
        "training_trials": len(training), "effective_training_weight": float(training["training_weight"].sum()),
        **{f"generated_positive_{r}": int(training.loc[training["label_source"].eq("generated"), r].sum())
           for r in REASONS},
        "original_training_sample_f1": original_training["metrics"]["sample_f1"],
        "validation_sample_f1": evaluation["metrics"]["sample_f1"],
        "validation_macro_f1": evaluation["metrics"]["macro_f1"],
        "delta_vs_baseline": evaluation["metrics"]["sample_f1"] - baseline_evaluation["metrics"]["sample_f1"],
        "validation_fallback_trials": evaluation["metrics"]["fallback_trials"],
        **{f"validation_f1_{r}": evaluation["metrics"][f"f1_{r}"] for r in REASONS},
        "training_dataset": str(dataset_path.relative_to(EXPERIMENT_DIR)),
        "kaggle_score": REPORTED_KAGGLE_SCORES.get(name, np.nan),
    }

In [14]:
baseline_training = labeled_train_df.copy()
baseline_training["label_source"] = "original"
baseline_training["training_weight"] = 1.0
baseline_dataset_path = EXPERIMENT_DIR / "datasets" / "baseline_train.csv"
baseline_training.to_csv(baseline_dataset_path, index=False)
save_validation_predictions(baseline_evaluation, EXPERIMENT_DIR / "validation_predictions" / "baseline.csv")
joblib.dump(refactored_baseline, EXPERIMENT_DIR / "models" / "baseline_C1000.joblib")

experiment_rows = [summarize_experiment(
    "baseline_C1000", "supervised", 0, baseline_training, refactored_baseline,
    baseline_evaluation, baseline_dataset_path,
)]
kmeans_experiments = {}

with threadpool_limits(limits=1):
    for k in K_VALUES:
        name = f"k{k:02d}"
        clustering = KMeans(n_clusters=k, **KMEANS_SETTINGS)
        cluster_ids = clustering.fit_predict(all_training_scaled)
        training, audit, clusters = make_augmented_training_data(train_df, cluster_ids, k)
        dataset_path = EXPERIMENT_DIR / "datasets" / f"{name}_train.csv"
        training.to_csv(dataset_path, index=False)
        audit.to_csv(EXPERIMENT_DIR / "label_audits" / f"{name}_labels.csv", index=False)
        clusters.to_csv(EXPERIMENT_DIR / "clusters" / f"{name}_clusters.csv", index=False)
        model = train_reason_classifier(
            training[EMBEDDING_COLUMNS].to_numpy(), training[REASONS].to_numpy(dtype=int),
            C=1000.0, sample_weight=training["training_weight"].to_numpy(),
        )
        evaluation = evaluate_reason_classifier(model, X_val, y_val)
        save_validation_predictions(evaluation, EXPERIMENT_DIR / "validation_predictions" / f"{name}.csv")
        joblib.dump({
            "classifier": model, "kmeans": clustering, "kmeans_scaler": kmeans_scaler,
            "reasons": REASONS, "embedding_columns": EMBEDDING_COLUMNS,
            "label_settings": LABEL_SETTINGS,
        }, EXPERIMENT_DIR / "models" / f"{name}.joblib")
        row = summarize_experiment(name, "kmeans_augmented", k, training, model, evaluation, dataset_path)
        row.update({
            "inertia": float(clustering.inertia_),
            "clusters_without_labeled_trials": int(clusters["labeled_trials"].eq(0).sum()),
            "clusters_eligible_for_generated_labels": int(clusters["eligible_for_generated_labels"].sum()),
        })
        experiment_rows.append(row)
        kmeans_experiments[k] = {"model": model, "clustering": clustering, "training": training,
                                 "audit": audit, "clusters": clusters, "evaluation": evaluation}
        print(f"K={k:2d}: {row['generated_trials']:4d} generated labels, "
              f"validation sample F1={row['validation_sample_f1']:.4f}, "
              f"macro F1={row['validation_macro_f1']:.4f}, "
              f"change={row['delta_vs_baseline']:+.4f}")

comparison = pd.DataFrame(experiment_rows)
comparison.to_csv(EXPERIMENT_DIR / "comparison.csv", index=False)
display(comparison[[
    "experiment", "generated_trials", "training_trials", "validation_sample_f1",
    "validation_macro_f1", "delta_vs_baseline", "validation_fallback_trials",
]].round(4))
display(comparison.set_index("experiment")[[f"validation_f1_{r}" for r in REASONS]].round(4))

K= 2:  388 generated labels, validation sample F1=0.8568, macro F1=0.6808, change=-0.0363
K= 5:  565 generated labels, validation sample F1=0.8547, macro F1=0.7024, change=-0.0385


K= 8:  591 generated labels, validation sample F1=0.7991, macro F1=0.5513, change=-0.0940


K=10:  706 generated labels, validation sample F1=0.8376, macro F1=0.5316, change=-0.0556


K=15:  662 generated labels, validation sample F1=0.8376, macro F1=0.6415, change=-0.0556


,experiment,generated_trials,training_trials,validation_sample_f1,validation_macro_f1,delta_vs_baseline,validation_fallback_trials
0,baseline_C1000,0,157,0.8932,0.8043,0.0000,4
1,k02,388,545,0.8568,0.6808,-0.0363,7
2,k05,565,722,0.8547,0.7024,-0.0385,9
3,k08,591,748,0.7991,0.5513,-0.0940,10
4,k10,706,863,0.8376,0.5316,-0.0556,5
5,k15,662,819,0.8376,0.6415,-0.0556,10


,validation_f1_Safety,validation_f1_Funding,validation_f1_Efficacy,validation_f1_Recruitment,validation_f1_Other
experiment,,,,,
baseline_C1000,0.9412,0.6667,0.8125,0.9697,0.6316
k02,0.8571,0.4000,0.7500,0.9524,0.4444
k05,0.9412,0.4000,0.7826,0.9174,0.4706
k08,0.6667,0.0000,0.7200,0.8991,0.4706
k10,0.6154,0.0000,0.7647,0.9703,0.3077
k15,0.6667,0.4000,0.8148,0.9259,0.4000


In [15]:
best_kmeans = comparison.loc[comparison["method"].eq("kmeans_augmented")].sort_values(
    ["validation_sample_f1", "k"], ascending=[False, True], kind="stable"
).iloc[0]
best_k = int(best_kmeans["k"])

# Check that the saved dataset, rather than hidden notebook state, reproduces the result.
saved_training = pd.read_csv(EXPERIMENT_DIR / best_kmeans["training_dataset"])
with threadpool_limits(limits=1):
    replayed_model = train_reason_classifier(
        saved_training[EMBEDDING_COLUMNS].to_numpy(), saved_training[REASONS].to_numpy(dtype=int),
        C=1000.0, sample_weight=saved_training["training_weight"].to_numpy(),
    )
replayed = evaluate_reason_classifier(replayed_model, X_val, y_val)
np.testing.assert_array_equal(replayed["predictions"], kmeans_experiments[best_k]["evaluation"]["predictions"])

# Independently verify that the first binary estimator received the requested weights.
weighted_model = kmeans_experiments[best_k]["model"]
scaled_saved = weighted_model.named_steps["scaler"].transform(saved_training[EMBEDDING_COLUMNS].to_numpy())
with threadpool_limits(limits=1):
    manual_binary = LogisticRegression(
        C=1000.0, solver="lbfgs", max_iter=5000, tol=1e-4, random_state=SEED,
    ).fit(scaled_saved, saved_training[REASONS[0]].to_numpy(), sample_weight=saved_training["training_weight"].to_numpy())
np.testing.assert_allclose(
    manual_binary.coef_, weighted_model.named_steps["classifier"].estimators_[0].coef_, rtol=1e-6, atol=1e-8
)

source_hashes = {
    name: hashlib.sha256(resolve_part1_input(DATA_DIR, name).read_bytes()).hexdigest()
    for name in ["part1_train.csv", "part1_val.csv", "submissions/archive/submitted/logistic_baseline__kaggle_0.87072.csv"]
}
manifest = {
    "features": EMBEDDING_COLUMNS, "pca_added": False,
    "k_values": K_VALUES, "kmeans_settings": KMEANS_SETTINGS, "label_settings": LABEL_SETTINGS,
    "classifier": {"type": "one-vs-rest logistic regression", "C": 1000.0,
                   "solver": "lbfgs", "max_iter": 5000, "tol": 1e-4, "random_state": SEED,
                   "scaler": "StandardScaler fitted unweighted on included training rows"},
    "prediction_rule": {"threshold": THRESHOLD, "fallback": "highest probability if no positive label"},
    "clustering_fit_rows": len(train_df), "known_training_rows": len(y_train),
    "validation_rows": len(y_val), "validation_used_for_fit": False,
    "best_kmeans_by_validation_sample_f1": best_k,
    "baseline_kaggle_score": {"score": 0.87072, "source": "reported by Michael after manual submission"},
    "reported_kaggle_scores": REPORTED_KAGGLE_SCORES,
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "source_sha256": source_hashes,
    "dataset_sha256": {str(p.relative_to(EXPERIMENT_DIR)): hashlib.sha256(p.read_bytes()).hexdigest()
                       for p in sorted((EXPERIMENT_DIR / "datasets").glob("*.csv"))},
}
(EXPERIMENT_DIR / "settings.json").write_text(json.dumps(manifest, indent=2) + "\n")

summary_lines = [
    "# K-means augmentation versus the supervised baseline", "",
    "All scores below use the same 78 validation trials and the same 0.5 threshold with a nonempty-label fallback.", "",
    "| Model | Generated trials | Training trials | Sample-wise F1 | Macro F1 | Change from baseline |",
    "|---|---:|---:|---:|---:|---:|",
]
for row in experiment_rows:
    summary_lines.append(
        f"| {row['experiment']} | {row['generated_trials']} | {row['training_trials']} | "
        f"{row['validation_sample_f1']:.4f} | {row['validation_macro_f1']:.4f} | {row['delta_vs_baseline']:+.4f} |"
    )
summary_lines.extend([
    "", "## Generated positive labels", "",
    "| K | Safety | Funding | Efficacy | Recruitment | Other |",
    "|---|---:|---:|---:|---:|---:|",
])
for row in experiment_rows[1:]:
    counts = " | ".join(str(row[f"generated_positive_{r}"]) for r in REASONS)
    summary_lines.append(f"| {row['k']} | {counts} |")
summary_lines.extend([
    "", "The generated labels are heavily skewed toward Recruitment. None of the five runs creates a positive "
    "Safety, Funding, or Other label. These generated trials therefore add negative targets for the rarer reasons. "
    "The reduced macro F1 and several reduced per-reason F1 scores are consistent with this imbalance; "
    "because the true hidden training labels are unavailable, the audit does not establish which generated labels are incorrect.",
    "", "## Per-reason validation F1", "",
    "| Model | Safety | Funding | Efficacy | Recruitment | Other |",
    "|---|---:|---:|---:|---:|---:|",
])
for row in experiment_rows:
    scores = " | ".join(f"{row[f'validation_f1_{r}']:.4f}" for r in REASONS)
    summary_lines.append(f"| {row['experiment']} | {scores} |")
summary_lines.extend([
    "", "## Interpretation", "",
    "The supervised baseline has the highest validation sample-wise and macro F1 among all six configurations. "
    "K=2 has a slightly higher reported Kaggle score, so retain both models as references. "
    "K=2 is the best K-means setting by the validation primary metric; "
    "K=5 has the best macro F1 among the K-means settings, but both remain below the supervised baseline. "
    "This result applies to the cluster-frequency labeling rule and fixed confidence/weight settings tested here; "
    "it does not show that every semi-supervised method will perform worse.",
])
summary_lines.extend([
    "", f"Best K-means configuration: **K={best_k}**, sample-wise F1 **{best_kmeans['validation_sample_f1']:.4f}**, "
    f"macro F1 **{best_kmeans['validation_macro_f1']:.4f}**.",
    f"Its sample-wise F1 difference from baseline is **{best_kmeans['delta_vs_baseline']:+.4f}**.",
    "", "The supervised Kaggle reference is **0.87072**, reported after Michael's manual submission. "
    "Michael reported **0.87393** for K=2 after manual submission, compared with **0.87072** for the supervised baseline. K=2 is lower on validation but higher on Kaggle by **0.00321** (0.321 F1 points). The ranking reversed; the Kaggle difference is small and does not establish a reliable advantage on future trials.",
    "", "Method: cluster all 1,171 training embeddings after training-only standardization, using 150 dimensions "
    "without additional PCA. Estimate cluster reason prevalences from the 157 known labels, smoothed toward "
    "their global prevalences with strength 2. Accept complete generated label vectors only with at least "
    "3 labeled cluster members, minimum assigned-label confidence 0.7, and at least one positive reason. "
    "Known labels receive weight 1 and generated labels weight 0.25. Fit C=1000 one-vs-rest logistic regression "
    "with a fresh scaler on each included training dataset. Only K varies across the five augmentation experiments.",
    "", "These confidence values are heuristic cluster frequencies. Validation selects among five settings, "
    "and the 78-trial validation set has already informed earlier supervised tuning. "
    "This comparison does not establish performance on an independent test set.",
    "", "Artifacts: `datasets/` contains the exact training rows and weights, `label_audits/` records all "
    "original training trials including excluded unknown labels, `clusters/` records support and reason frequencies, "
    "`models/` stores fitted pipelines and cluster models, and `validation_predictions/` stores probabilities, "
    "labels, and fallback use. `comparison.csv` includes per-reason F1, and `settings.json` records settings and checksums.",
    "", "Verification: refactoring reproduced baseline probabilities and predictions; original known labels "
    "were preserved; validation IDs were excluded from augmented training; the saved winning K-means dataset "
    "reproduced its predictions; and an independently fitted binary classifier confirmed sample weights were routed correctly.",
])
summary_text = "\n".join(summary_lines) + "\n"
(EXPERIMENT_DIR / "SUMMARY.md").write_text(summary_text)
print(summary_text)
print("Artifacts saved under:", EXPERIMENT_DIR)

# K-means augmentation versus the supervised baseline

All scores below use the same 78 validation trials and the same 0.5 threshold with a nonempty-label fallback.

| Model | Generated trials | Training trials | Sample-wise F1 | Macro F1 | Change from baseline |
|---|---:|---:|---:|---:|---:|
| baseline_C1000 | 0 | 157 | 0.8932 | 0.8043 | +0.0000 |
| k02 | 388 | 545 | 0.8568 | 0.6808 | -0.0363 |
| k05 | 565 | 722 | 0.8547 | 0.7024 | -0.0385 |
| k08 | 591 | 748 | 0.7991 | 0.5513 | -0.0940 |
| k10 | 706 | 863 | 0.8376 | 0.5316 | -0.0556 |
| k15 | 662 | 819 | 0.8376 | 0.6415 | -0.0556 |

## Generated positive labels

| K | Safety | Funding | Efficacy | Recruitment | Other |
|---|---:|---:|---:|---:|---:|
| 2 | 0 | 0 | 0 | 388 | 0 |
| 5 | 0 | 0 | 0 | 565 | 0 |
| 8 | 0 | 0 | 0 | 591 | 0 |
| 10 | 0 | 0 | 128 | 578 | 0 |
| 15 | 0 | 0 | 31 | 631 | 0 |

The generated labels are heavily skewed toward Recruitment. None of the five runs creates a positive Safety, Funding, or Other label. These ge

## Export K=2 for the Kaggle comparison

Load the saved K=2 experiment's logistic regression classifier. It was fitted on 157 original labeled trials and 388 generated-label trials (545 total), with weights 1 and 0.25 respectively. Do not refit on validation or test data. The classifier's fitted scaler handles the raw test embeddings.

Use exactly the supervised submission's prediction rule: select probabilities >= 0.5 and use the highest-probability reason if no reason qualifies. Verify that the loaded model reproduces its recorded validation score, then export a separate 312-row CSV in the sample submission's ID order. Upload this file manually to Part 1; the reported Kaggle result is recorded in the final section.

In [16]:
k2_model_path = EXPERIMENT_DIR / "models" / "k02.joblib"
k2_saved = joblib.load(k2_model_path)
k2_classifier = k2_saved["classifier"]
assert k2_saved["reasons"] == REASONS
assert k2_classifier.named_steps["classifier"].estimator.C == 1000.0
k2_training = pd.read_csv(EXPERIMENT_DIR / "datasets" / "k02_train.csv")
assert len(k2_training) == k2_classifier.named_steps["scaler"].n_samples_seen_ == 545
assert k2_training["label_source"].eq("original").sum() == 157
assert k2_training["label_source"].eq("generated").sum() == 388

k2_validation = evaluate_reason_classifier(k2_classifier, X_val, y_val)
k2_recorded = comparison.set_index("experiment").loc["k02"]
np.testing.assert_allclose(
    k2_validation["metrics"]["sample_f1"], k2_recorded["validation_sample_f1"], atol=1e-14
)
np.testing.assert_allclose(
    k2_validation["metrics"]["macro_f1"], k2_recorded["validation_macro_f1"], atol=1e-14
)

k2_test = pd.read_csv(resolve_part1_input(DATA_DIR, "part1_test.csv"))
k2_sample = pd.read_csv(resolve_part1_input(DATA_DIR, "part1_sample_submission.csv"))
assert k2_test["ID"].is_unique and k2_sample["ID"].is_unique
assert set(k2_test["ID"]) == set(k2_sample["ID"])
assert set(k2_test["ID"]).isdisjoint(k2_training["ID"])
assert set(k2_test["ID"]).isdisjoint(val_df["ID"])
assert np.isfinite(k2_test[EMBEDDING_COLUMNS].to_numpy()).all()
k2_probabilities = k2_classifier.predict_proba(k2_test[EMBEDDING_COLUMNS].to_numpy())
k2_predictions, k2_fallback = predict_reason_labels(k2_probabilities, THRESHOLD, ensure_nonempty=True)
k2_submission = pd.DataFrame({
    "ID": k2_test["ID"],
    "Predicted": [" ".join(r for r, yes in zip(REASONS, row) if yes) for row in k2_predictions],
})
k2_submission = k2_submission.set_index("ID").loc[k2_sample["ID"]].reset_index()
assert k2_submission["ID"].tolist() == k2_sample["ID"].tolist()
assert k2_submission.columns.tolist() == ["ID", "Predicted"]
assert len(k2_submission) == 312 and k2_submission["Predicted"].str.len().gt(0).all()
assert k2_submission["Predicted"].map(lambda s: set(s.split()).issubset(REASONS)).all()
k2_submission_path = DATA_DIR / "submissions/archive/submitted/kmeans_k2__kaggle_0.87393.csv"
k2_submission.to_csv(k2_submission_path, index=False)
pd.testing.assert_frame_equal(pd.read_csv(k2_submission_path), k2_submission)

k2_submission_record = {
    "experiment": "k02", "training_trials": len(k2_training),
    "original_trials": 157, "generated_trials": 388, "classifier_C": 1000.0,
    "generated_label_weight": 0.25, "threshold": THRESHOLD,
    "fallback": "highest-probability reason when no reason meets threshold",
    "validation_sample_f1": k2_validation["metrics"]["sample_f1"],
    "validation_macro_f1": k2_validation["metrics"]["macro_f1"],
    "test_rows": len(k2_submission), "test_fallback_trials": int(k2_fallback.sum()),
    "submission_file": str(k2_submission_path.relative_to(DATA_DIR)),
    "submission_sha256": hashlib.sha256(k2_submission_path.read_bytes()).hexdigest(),
    "model_file": "models/k02.joblib",
    "model_sha256": hashlib.sha256(k2_model_path.read_bytes()).hexdigest(),
    "kaggle_score": REPORTED_KAGGLE_SCORES["k02"],
    "kaggle_status": "submitted; score reported by Michael",
    "baseline_kaggle_score": 0.87072,
}
(EXPERIMENT_DIR / "k02_submission.json").write_text(json.dumps(k2_submission_record, indent=2) + "\n")

k2_summary_path = EXPERIMENT_DIR / "SUMMARY.md"
k2_summary = k2_summary_path.read_text().split("## K=2 Kaggle submission")[0].rstrip()
k2_summary += (
    "\n\n## K=2 Kaggle submission\n\n"
    f"Prepared `{k2_submission_path.name}` with 312 predictions from the saved K=2 classifier, "
    f"trained on 157 known-label and 388 generated-label trials. Test fallback cases: {int(k2_fallback.sum())}. "
    "The prediction rule matches the supervised baseline. "
    "**Kaggle score: 0.87393 (reported by Michael after manual submission).** Baseline Kaggle reference: **0.87072**. "
    "`k02_submission.json` records the model and submission checksums.\n"
)
k2_summary_path.write_text(k2_summary)
print(f"Saved {len(k2_submission)} predictions to: {k2_submission_path.resolve()}")
print(f"Validation sample-wise F1: {k2_validation['metrics']['sample_f1']:.4f}")
print(f"Validation macro F1: {k2_validation['metrics']['macro_f1']:.4f}")
print(f"Test trials requiring fallback: {k2_fallback.sum()}")
display(k2_submission.head(10))

Saved 312 predictions to: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/submissions/archive/submitted/kmeans_k2__kaggle_0.87393.csv
Validation sample-wise F1: 0.8568
Validation macro F1: 0.6808
Test trials requiring fallback: 25


,ID,Predicted
0,TR00756,Efficacy
1,TR05267,Safety
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety
5,TR01460,Recruitment
6,TR00199,Recruitment
7,TR09959,Recruitment
8,TR01886,Recruitment
9,TR09553,Other


## Reported Kaggle comparison: K=2 versus supervised baseline

Michael reported the following scores after manually uploading the two saved submission files:

| Model | Validation sample-wise F1 | Kaggle score |
|---|---:|---:|
| Supervised C=1000, with fallback | 0.8932 | 0.87072 |
| K-means K=2 augmentation, C=1000, with fallback | 0.8568 | 0.87393 |

The K=2 model is lower on validation by approximately 0.03632 but higher on Kaggle by **0.00321**, or **0.321 F1 points**. The scores are in a similar range, but the ranking reversed: validation did not correctly predict the Kaggle winner in this comparison. The small validation set and later-year test distribution can lead to such differences. Without trial-level test labels, these aggregate scores do not establish a statistically reliable improvement.

Retain both as references: the supervised model leads validation, while K=2 has the higher observed Kaggle score. These external scores are user-reported rather than independently retrieved. The label-distribution concerns remain useful diagnostics; they did not establish that K=2 would have worse Kaggle performance.


# Year as a classifier feature

Compare embeddings alone with embeddings plus standardized `start_year`, separately for the supervised model and the K=2 augmented model. These are four configurations: two existing references and two new classifiers with 151 input features.

Reuse the exact saved training rows, reason labels, and weights. Load the existing 150-feature classifiers as references. Add year only to the final classifier: K-means still used 150 embeddings, and no cluster assignment or generated label changes. Keep C=1000, L2 regularization, solver settings, the 0.5 threshold, and the nonempty-label fallback fixed. Each new classifier learns its embedding and year standardization on its own training rows only.

Use the same 78 validation trials and save each model, the exact selected feature table, predictions, and settings. The existing Kaggle scores apply only to the original models. The year-feature models have no Kaggle result yet.

In [17]:
# This section loads saved experiment inputs; it does not need to regenerate labels.
import hashlib
import json
import platform
import joblib
import sklearn
from threadpoolctl import threadpool_limits

YEAR_FEATURE_COLUMNS = EMBEDDING_COLUMNS + ["start_year"]
# External scores reported by Michael after manual submissions.
YEAR_REPORTED_KAGGLE_SCORES = {"baseline_embeddings": 0.87072, "k02_embeddings": 0.87393, "k02_embeddings_year": 0.85149}
def resolve_part1_input(directory, name):
    """Locate supplied CSV inputs while leaving experiment outputs under part1/."""
    supplied = {"part1_train.csv", "part1_val.csv", "part1_test.csv", "part1_sample_submission.csv", "pathway_catalog.csv"}
    nested = directory / "data" / name
    return nested if name in supplied and nested.is_file() else directory / name

year_cwd = Path.cwd()
year_candidates = [year_cwd, year_cwd / "part1", year_cwd / "AML-Midterm" / "part1"]
year_data_dir = next(d for d in year_candidates if ((d / "part1_train.csv").is_file() or (d / "data" / "part1_train.csv").is_file())).resolve()
year_source_dir = year_data_dir / "experiments" / "kmeans"
YEAR_EXPERIMENT_DIR = year_data_dir / "experiments" / "year_feature"
for folder in ["datasets", "models", "validation_predictions"]:
    (YEAR_EXPERIMENT_DIR / folder).mkdir(parents=True, exist_ok=True)

year_val_df = pd.read_csv(resolve_part1_input(year_data_dir, "part1_val.csv"))
year_val_labels = year_val_df[REASONS].to_numpy(dtype=int)
assert len(year_val_df) == 78
assert np.isfinite(year_val_df[YEAR_FEATURE_COLUMNS].to_numpy()).all()
year_original_train = pd.read_csv(resolve_part1_input(year_data_dir, "part1_train.csv"))
year_known_mask = year_original_train[REASONS].isin([0, 1]).all(axis=1)
year_original_labeled = year_original_train.loc[year_known_mask].copy()
year_reference_scores = pd.read_csv(year_source_dir / "comparison.csv").set_index("experiment")
year_source_manifest = json.loads((year_source_dir / "settings.json").read_text())

year_training_sets = {
    "baseline": pd.read_csv(year_source_dir / "datasets" / "baseline_train.csv"),
    "k02": pd.read_csv(year_source_dir / "datasets" / "k02_train.csv"),
}
year_reference_models = {
    "baseline": joblib.load(year_source_dir / "models" / "baseline_C1000.joblib"),
    "k02": joblib.load(year_source_dir / "models" / "k02.joblib")["classifier"],
}
year_frozen_paths = [
    year_source_dir / "datasets" / "baseline_train.csv",
    year_source_dir / "datasets" / "k02_train.csv",
    year_source_dir / "label_audits" / "k02_labels.csv",
    year_source_dir / "clusters" / "k02_clusters.csv",
    year_source_dir / "models" / "k02.joblib",
    year_data_dir / "submissions/archive/submitted/logistic_baseline__kaggle_0.87072.csv",
    year_data_dir / "submissions/archive/submitted/kmeans_k2__kaggle_0.87393.csv",
]
year_frozen_hashes = {str(q.relative_to(year_data_dir)): hashlib.sha256(q.read_bytes()).hexdigest()
                      for q in year_frozen_paths}
for family, training in year_training_sets.items():
    relative = f"datasets/{'baseline' if family == 'baseline' else 'k02'}_train.csv"
    assert hashlib.sha256((year_source_dir / relative).read_bytes()).hexdigest() == year_source_manifest["dataset_sha256"][relative]
    assert training["ID"].is_unique
    assert set(training["ID"]).isdisjoint(year_val_df["ID"])
    assert training[REASONS].isin([0, 1]).all().all()
    assert np.isfinite(training[YEAR_FEATURE_COLUMNS].to_numpy()).all()
    known_rows = training.loc[training["label_source"].eq("original")].set_index("ID")
    pd.testing.assert_frame_equal(known_rows[REASONS], year_original_labeled.set_index("ID").loc[known_rows.index, REASONS])
assert len(year_training_sets["baseline"]) == 157
assert len(year_training_sets["k02"]) == 545
assert year_training_sets["k02"].loc[lambda d: d.label_source.eq("generated"), "training_weight"].eq(0.25).all()
print("Loaded frozen training datasets: baseline=157 trials, K=2=545 trials (157 original + 388 generated).")

Loaded frozen training datasets: baseline=157 trials, K=2=545 trials (157 original + 388 generated).


In [18]:
year_rows = []
year_models = {}
year_evaluations = {}
year_coefficient_rows = []
year_artifact_hashes = {}

with threadpool_limits(limits=1):
    for family, training in year_training_sets.items():
        reference_name = "baseline_C1000" if family == "baseline" else "k02"
        for include_year in [False, True]:
            name = f"{family}_embeddings" + ("_year" if include_year else "")
            features = YEAR_FEATURE_COLUMNS if include_year else EMBEDDING_COLUMNS
            if include_year:
                model = train_reason_classifier(
                    training[features].to_numpy(), training[REASONS].to_numpy(dtype=int), C=1000.0,
                    sample_weight=None if family == "baseline" else training["training_weight"].to_numpy(),
                )
            else:
                model = year_reference_models[family]
            evaluation = evaluate_reason_classifier(model, year_val_df[features].to_numpy(), year_val_labels)
            if not include_year:
                np.testing.assert_allclose(evaluation["metrics"]["sample_f1"], year_reference_scores.loc[reference_name, "validation_sample_f1"], atol=1e-14)
                np.testing.assert_allclose(evaluation["metrics"]["macro_f1"], year_reference_scores.loc[reference_name, "validation_macro_f1"], atol=1e-14)
            assert model.named_steps["scaler"].n_features_in_ == len(features)
            assert model.named_steps["scaler"].n_samples_seen_ == len(training)

            selected_training = training[["ID"] + features + REASONS + ["label_source", "training_weight"]].copy()
            dataset_path = YEAR_EXPERIMENT_DIR / "datasets" / f"{name}_train.csv"
            selected_training.to_csv(dataset_path, index=False)
            year_artifact_hashes[str(dataset_path.relative_to(YEAR_EXPERIMENT_DIR))] = hashlib.sha256(dataset_path.read_bytes()).hexdigest()
            output = year_val_df[["ID"] + REASONS].rename(columns={r: f"actual_{r}" for r in REASONS}).copy()
            for i, reason in enumerate(REASONS):
                output[f"probability_{reason}"] = evaluation["probabilities"][:, i]
                output[f"predicted_{reason}"] = evaluation["predictions"][:, i]
            output["used_fallback"] = evaluation["fallback"]
            output.to_csv(YEAR_EXPERIMENT_DIR / "validation_predictions" / f"{name}.csv", index=False)
            joblib.dump({"classifier": model, "feature_columns": features, "reasons": REASONS,
                         "family": family, "year_feature": include_year},
                        YEAR_EXPERIMENT_DIR / "models" / f"{name}.joblib")
            original_training_eval = evaluate_reason_classifier(
                model, year_original_labeled[features].to_numpy(), year_original_labeled[REASONS].to_numpy(dtype=int)
            )
            row = {
                "experiment": name, "family": family, "year_feature": include_year,
                "features": len(features), "C": 1000.0, "training_trials": len(training),
                "original_trials": int(training.label_source.eq("original").sum()),
                "generated_trials": int(training.label_source.eq("generated").sum()),
                "effective_training_weight": float(training.training_weight.sum()),
                "original_training_sample_f1": original_training_eval["metrics"]["sample_f1"],
                "validation_sample_f1": evaluation["metrics"]["sample_f1"],
                "validation_macro_f1": evaluation["metrics"]["macro_f1"],
                "delta_vs_same_family_without_year": evaluation["metrics"]["sample_f1"] - year_reference_scores.loc[reference_name, "validation_sample_f1"],
                "validation_fallback_trials": evaluation["metrics"]["fallback_trials"],
                **{f"validation_f1_{r}": evaluation["metrics"][f"f1_{r}"] for r in REASONS},
                "kaggle_score": YEAR_REPORTED_KAGGLE_SCORES.get(name, np.nan),
                "training_dataset": str(dataset_path.relative_to(YEAR_EXPERIMENT_DIR)),
            }
            year_rows.append(row)
            year_models[name] = model
            year_evaluations[name] = evaluation
            if include_year:
                year_index = features.index("start_year")
                np.testing.assert_allclose(model.named_steps["scaler"].mean_[year_index], training.start_year.mean())
                for reason, estimator in zip(REASONS, model.named_steps["classifier"].estimators_):
                    year_coefficient_rows.append({
                        "experiment": name, "reason": reason,
                        "standardized_year_coefficient": float(estimator.coef_[0, year_index]),
                        "training_year_mean": float(model.named_steps["scaler"].mean_[year_index]),
                        "training_year_std": float(model.named_steps["scaler"].scale_[year_index]),
                    })
            print(f"{name}: sample F1={row['validation_sample_f1']:.4f}, "
                  f"macro F1={row['validation_macro_f1']:.4f}, "
                  f"year effect={row['delta_vs_same_family_without_year']:+.4f}")

year_comparison = pd.DataFrame(year_rows)
year_comparison.to_csv(YEAR_EXPERIMENT_DIR / "comparison.csv", index=False)
year_coefficients = pd.DataFrame(year_coefficient_rows)
year_coefficients.to_csv(YEAR_EXPERIMENT_DIR / "year_coefficients.csv", index=False)
display(year_comparison[["experiment", "features", "validation_sample_f1", "validation_macro_f1",
                         "validation_f1_Safety", "delta_vs_same_family_without_year", "kaggle_score"]].round(4))
display(year_comparison.set_index("experiment")[[f"validation_f1_{r}" for r in REASONS]].round(4))

baseline_embeddings: sample F1=0.8932, macro F1=0.8043, year effect=+0.0000
baseline_embeddings_year: sample F1=0.8974, macro F1=0.8166, year effect=+0.0043


k02_embeddings: sample F1=0.8568, macro F1=0.6808, year effect=+0.0000


k02_embeddings_year: sample F1=0.8611, macro F1=0.6937, year effect=+0.0043


,experiment,features,validation_sample_f1,validation_macro_f1,validation_f1_Safety,delta_vs_same_family_without_year,kaggle_score
0,baseline_embeddings,150,0.8932,0.8043,0.9412,0.0000,0.8707
1,baseline_embeddings_year,151,0.8974,0.8166,0.9412,0.0043,NaN
2,k02_embeddings,150,0.8568,0.6808,0.8571,0.0000,0.8739
3,k02_embeddings_year,151,0.8611,0.6937,0.8571,0.0043,0.8515


,validation_f1_Safety,validation_f1_Funding,validation_f1_Efficacy,validation_f1_Recruitment,validation_f1_Other
experiment,,,,,
baseline_embeddings,0.9412,0.6667,0.8125,0.9697,0.6316
baseline_embeddings_year,0.9412,0.6667,0.8387,0.9697,0.6667
k02_embeddings,0.8571,0.4000,0.7500,0.9524,0.4444
k02_embeddings_year,0.8571,0.4000,0.7500,0.9615,0.5000


In [19]:
# Verify that exported inputs and saved models reproduce the year-feature results.
with threadpool_limits(limits=1):
    for family in ["baseline", "k02"]:
        name = f"{family}_embeddings_year"
        row = year_comparison.set_index("experiment").loc[name]
        saved_training = pd.read_csv(YEAR_EXPERIMENT_DIR / row["training_dataset"])
        original_training = year_training_sets[family]
        pd.testing.assert_frame_equal(
            saved_training[["ID"] + REASONS + ["label_source", "training_weight"]],
            original_training[["ID"] + REASONS + ["label_source", "training_weight"]],
        )
        replay = train_reason_classifier(
            saved_training[YEAR_FEATURE_COLUMNS].to_numpy(), saved_training[REASONS].to_numpy(dtype=int),
            C=1000.0, sample_weight=None if family == "baseline" else saved_training.training_weight.to_numpy(),
        )
        replay_eval = evaluate_reason_classifier(replay, year_val_df[YEAR_FEATURE_COLUMNS].to_numpy(), year_val_labels)
        np.testing.assert_array_equal(replay_eval["predictions"], year_evaluations[name]["predictions"])
        reloaded = joblib.load(YEAR_EXPERIMENT_DIR / "models" / f"{name}.joblib")
        np.testing.assert_allclose(reloaded["classifier"].predict_proba(year_val_df[YEAR_FEATURE_COLUMNS].to_numpy()),
                                   year_evaluations[name]["probabilities"], rtol=1e-12, atol=1e-12)
for file_name, expected_hash in year_frozen_hashes.items():
    assert hashlib.sha256((year_data_dir / file_name).read_bytes()).hexdigest() == expected_hash

year_settings = {
    "question": "Effect of adding standardized start_year to final classifiers only",
    "embedding_features": EMBEDDING_COLUMNS, "year_features": YEAR_FEATURE_COLUMNS,
    "clustering_unchanged": True, "generated_labels_unchanged": True,
    "classifier_C": 1000.0, "solver": "lbfgs", "max_iter": 5000, "tol": 1e-4, "random_state": SEED,
    "scaling": "StandardScaler fitted unweighted on each model's own training rows",
    "original_weight": 1.0, "generated_weight": 0.25,
    "threshold": 0.5, "fallback": "highest-probability reason if no label reaches threshold",
    "validation_trials": len(year_val_df), "validation_used_for_fit": False,
    "frozen_input_sha256": year_frozen_hashes, "dataset_sha256": year_artifact_hashes,
    "validation_sha256": hashlib.sha256((resolve_part1_input(year_data_dir, "part1_val.csv")).read_bytes()).hexdigest(),
    "versions": {"python": platform.python_version(), "sklearn": sklearn.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "kaggle_scores": YEAR_REPORTED_KAGGLE_SCORES,
    "kaggle_provenance": "Scores reported by Michael after manual submissions; supervised baseline plus year remains unsubmitted",
}
(YEAR_EXPERIMENT_DIR / "settings.json").write_text(json.dumps(year_settings, indent=2) + "\n")

year_summary_lines = [
    "# Year as a classifier feature", "",
    "Controlled comparison on the same 78 validation trials. Add only standardized start_year to the final "
    "classifier; use the same training rows, labels, weights, C=1000, threshold 0.5, and nonempty fallback.", "",
    "| Model | Features | Sample-wise F1 | Macro F1 | Safety F1 | Effect of year | Kaggle |",
    "|---|---:|---:|---:|---:|---:|---:|",
]
for row in year_rows:
    score = "unsubmitted" if pd.isna(row["kaggle_score"]) else f"{row['kaggle_score']:.5f}"
    year_summary_lines.append(
        f"| {row['experiment']} | {row['features']} | {row['validation_sample_f1']:.4f} | "
        f"{row['validation_macro_f1']:.4f} | {row['validation_f1_Safety']:.4f} | "
        f"{row['delta_vs_same_family_without_year']:+.4f} | {score} |"
    )
year_summary_lines.extend(["", "## Results", ""])
for family in ["baseline", "k02"]:
    without = year_comparison.set_index("experiment").loc[f"{family}_embeddings"]
    added = year_comparison.set_index("experiment").loc[f"{family}_embeddings_year"]
    effect = added.validation_sample_f1 - without.validation_sample_f1
    direction = "improved" if effect > 0 else "reduced" if effect < 0 else "did not change"
    year_summary_lines.append(
        f"Adding year {direction} {family} validation sample-wise F1 from {without.validation_sample_f1:.4f} "
        f"to {added.validation_sample_f1:.4f} ({effect:+.4f}). Safety F1 changed from "
        f"{without.validation_f1_Safety:.4f} to {added.validation_f1_Safety:.4f}."
    )
year_summary_lines.extend([
    "", "## Per-reason validation F1", "",
    "| Model | Safety | Funding | Efficacy | Recruitment | Other |",
    "|---|---:|---:|---:|---:|---:|",
])
for row in year_rows:
    values = " | ".join(f"{row[f'validation_f1_{r}']:.4f}" for r in REASONS)
    year_summary_lines.append(f"| {row['experiment']} | {values} |")
year_summary_lines.extend([
    "", "## Interpretation and artifacts", "",
    "K-means still used 150 embeddings. The K=2 training set remains exactly 157 original-label trials "
    "plus 388 generated-label trials with generated weight 0.25. No year-based Safety probability or label "
    "was manually imposed. Year's influence was learned through the same regularized classifiers. "
    "The supplied labeled subset was deliberately enriched for rare reasons, so year coefficients are "
    "associations in the selected training data and must not be interpreted as unbiased population trends.",
    "", "`year_coefficients.csv` records each fitted year coefficient per one training standard deviation of year, "
    "holding the embedding features fixed. Positive coefficients increase that reason's predicted log-odds; "
    "negative coefficients decrease them. The scaling mean and standard deviation are recorded for each model.",
    "", "`datasets/` saves the exact selected features, labels, and weights for all four configurations. "
    "`models/` saves fitted pipelines with their feature lists; `validation_predictions/` saves per-trial results. "
    "`comparison.csv` contains metrics and `settings.json` contains settings and checksums. "
    "Saved datasets reproduce the year models' predictions, loaded models reproduce their probabilities, "
    "and the source datasets, label audit, clusters, K=2 model, and both existing submission CSVs remain unchanged.",
    "", "Existing Kaggle scores are user-reported. The K=2 year-feature submission is exported in the following section. "
    "K=2 plus year scored 0.85149 after manual submission, below K=2 without year at 0.87393. The supervised baseline plus year remains unsubmitted. These validation results use the same small set that informed earlier experiments.",
])
year_summary_lines.append('## Reported Kaggle result: adding year to K=2\n\nMichael reported **0.85149** after submitting `submissions/archive/submitted/kmeans_k2_year__kaggle_0.85149.csv`.\n\n| K=2 classifier | Validation sample-wise F1 | Kaggle score |\n|---|---:|---:|\n| Embeddings only | 0.8568 | **0.87393** |\n| Embeddings plus year | **0.8611** | 0.85149 |\n\nAdding year improved validation by approximately 0.00427 but reduced the reported Kaggle score by **0.02244**, or **2.244 F1 points**. Retain K=2 without year as the best observed Kaggle model among the submitted experiments. This year-feature change did not transfer successfully to the scored test trials.\n\nA plausible explanation is that year relationships learned from the selected/enriched labels and earlier trials do not transfer to later trials. The aggregate Kaggle score alone cannot establish that cause. This result applies to year as a feature in the K=2 augmented classifier; the supervised model plus year remains unsubmitted, and year has not been added to clustering. Preserve both models and record this negative result.\n')
year_summary = "\n".join(year_summary_lines) + "\n"
(YEAR_EXPERIMENT_DIR / "SUMMARY.md").write_text(year_summary)
print(year_summary)
print("Artifacts saved under:", YEAR_EXPERIMENT_DIR)

# Year as a classifier feature

Controlled comparison on the same 78 validation trials. Add only standardized start_year to the final classifier; use the same training rows, labels, weights, C=1000, threshold 0.5, and nonempty fallback.

| Model | Features | Sample-wise F1 | Macro F1 | Safety F1 | Effect of year | Kaggle |
|---|---:|---:|---:|---:|---:|---:|
| baseline_embeddings | 150 | 0.8932 | 0.8043 | 0.9412 | +0.0000 | 0.87072 |
| baseline_embeddings_year | 151 | 0.8974 | 0.8166 | 0.9412 | +0.0043 | unsubmitted |
| k02_embeddings | 150 | 0.8568 | 0.6808 | 0.8571 | +0.0000 | 0.87393 |
| k02_embeddings_year | 151 | 0.8611 | 0.6937 | 0.8571 | +0.0043 | 0.85149 |

## Results

Adding year improved baseline validation sample-wise F1 from 0.8932 to 0.8974 (+0.0043). Safety F1 changed from 0.9412 to 0.9412.
Adding year improved k02 validation sample-wise F1 from 0.8568 to 0.8611 (+0.0043). Safety F1 changed from 0.8571 to 0.8571.

## Per-reason validation F1

| Model | Safety | Funding | 

## Export K=2 plus year for Kaggle

Load the saved `k02_embeddings_year` classifier, trained on the same 157 original-label and 388 generated-label trials. Use its 151 features in the saved order, including start_year, and its fitted scaler. Keep C=1000, threshold 0.5, and the highest-probability fallback for empty predictions.

Verify its recorded validation scores before exporting a separate 312-row submission. Compare the manually returned Kaggle score against **0.87393**, the K=2 model without year. The reported Kaggle result is recorded in the following section.

In [20]:
k2_year_model_path = YEAR_EXPERIMENT_DIR / "models" / "k02_embeddings_year.joblib"
k2_year_saved = joblib.load(k2_year_model_path)
k2_year_classifier = k2_year_saved["classifier"]
k2_year_features = k2_year_saved["feature_columns"]
assert k2_year_features == EMBEDDING_COLUMNS + ["start_year"]
assert k2_year_saved["reasons"] == REASONS
assert k2_year_classifier.named_steps["scaler"].n_features_in_ == 151
assert k2_year_classifier.named_steps["scaler"].n_samples_seen_ == 545
assert k2_year_classifier.named_steps["classifier"].estimator.C == 1000.0

k2_year_val = evaluate_reason_classifier(
    k2_year_classifier, year_val_df[k2_year_features].to_numpy(), year_val_labels,
    threshold=0.5, ensure_nonempty=True,
)
k2_year_reference = year_comparison.set_index("experiment").loc["k02_embeddings_year"]
np.testing.assert_allclose(k2_year_val["metrics"]["sample_f1"], k2_year_reference.validation_sample_f1, atol=1e-14)
np.testing.assert_allclose(k2_year_val["metrics"]["macro_f1"], k2_year_reference.validation_macro_f1, atol=1e-14)

k2_year_test = pd.read_csv(resolve_part1_input(year_data_dir, "part1_test.csv"))
k2_year_sample = pd.read_csv(resolve_part1_input(year_data_dir, "part1_sample_submission.csv"))
assert k2_year_test["ID"].is_unique and k2_year_sample["ID"].is_unique
assert set(k2_year_test["ID"]) == set(k2_year_sample["ID"])
assert set(k2_year_test["ID"]).isdisjoint(year_training_sets["k02"]["ID"])
assert set(k2_year_test["ID"]).isdisjoint(year_val_df["ID"])
assert np.isfinite(k2_year_test[k2_year_features].to_numpy()).all()
k2_year_probabilities = k2_year_classifier.predict_proba(k2_year_test[k2_year_features].to_numpy())
k2_year_predicted, k2_year_fallback = predict_reason_labels(k2_year_probabilities, threshold=0.5, ensure_nonempty=True)
k2_year_submission = pd.DataFrame({
    "ID": k2_year_test["ID"],
    "Predicted": [" ".join(r for r, yes in zip(REASONS, row) if yes) for row in k2_year_predicted],
}).set_index("ID").loc[k2_year_sample["ID"]].reset_index()
assert k2_year_submission.shape == (312, 2)
assert k2_year_submission["ID"].tolist() == k2_year_sample["ID"].tolist()
assert k2_year_submission["Predicted"].str.len().gt(0).all()
assert k2_year_submission["Predicted"].map(lambda s: set(s.split()).issubset(REASONS)).all()
k2_year_submission_path = year_data_dir / "submissions/archive/submitted/kmeans_k2_year__kaggle_0.85149.csv"
k2_year_submission.to_csv(k2_year_submission_path, index=False)
pd.testing.assert_frame_equal(pd.read_csv(k2_year_submission_path), k2_year_submission)

k2_without_year = pd.read_csv(year_data_dir / "submissions/archive/submitted/kmeans_k2__kaggle_0.87393.csv").set_index("ID")
k2_year_changed = int(sum(
    set(row.Predicted.split()) != set(k2_without_year.loc[row.ID, "Predicted"].split())
    for row in k2_year_submission.itertuples(index=False)
))
k2_year_record = {
    "experiment": "k02_embeddings_year", "features": k2_year_features, "classifier_C": 1000.0,
    "original_trials": 157, "generated_trials": 388, "generated_weight": 0.25,
    "threshold": 0.5, "fallback": "highest-probability reason if no label reaches threshold",
    "validation_sample_f1": k2_year_val["metrics"]["sample_f1"],
    "validation_macro_f1": k2_year_val["metrics"]["macro_f1"],
    "test_rows": len(k2_year_submission), "test_fallback_trials": int(k2_year_fallback.sum()),
    "test_trials_changed_from_without_year": k2_year_changed,
    "submission_file": str(k2_year_submission_path.relative_to(year_data_dir)),
    "submission_sha256": hashlib.sha256(k2_year_submission_path.read_bytes()).hexdigest(),
    "model_file": "models/k02_embeddings_year.joblib",
    "model_sha256": hashlib.sha256(k2_year_model_path.read_bytes()).hexdigest(),
    "kaggle_score": YEAR_REPORTED_KAGGLE_SCORES["k02_embeddings_year"],
    "kaggle_status": "submitted; score reported by Michael",
    "kaggle_change_from_without_year": -0.02244,
    "kaggle_comparator_without_year": 0.87393,
}
(YEAR_EXPERIMENT_DIR / "k02_year_submission.json").write_text(json.dumps(k2_year_record, indent=2) + "\n")
k2_year_summary_path = YEAR_EXPERIMENT_DIR / "SUMMARY.md"
k2_year_summary = k2_year_summary_path.read_text().split("## K=2 plus year Kaggle submission")[0].rstrip()
k2_year_summary += (
    "\n\n## K=2 plus year Kaggle submission\n\n"
    f"Prepared `{k2_year_submission_path.name}` with 312 predictions from the saved 151-feature classifier. "
    f"Test trials using the nonempty fallback: {int(k2_year_fallback.sum())}. "
    f"Predictions differ from K=2 without year on {k2_year_changed} trials. "
    "**Kaggle result: 0.85149 (reported by Michael after manual submission).** Compare with K=2 without year at **0.87393**. "
    "The model and submission checksums are recorded in `k02_year_submission.json`.\n"
)
k2_year_summary_path.write_text(k2_year_summary)
print(f"Saved {len(k2_year_submission)} predictions to: {k2_year_submission_path}")
print(f"Validation sample-wise F1: {k2_year_val['metrics']['sample_f1']:.4f}")
print(f"Validation macro F1: {k2_year_val['metrics']['macro_f1']:.4f}")
print(f"Test trials changed from K=2 without year: {k2_year_changed}")
print(f"Test trials requiring fallback: {k2_year_fallback.sum()}")
display(k2_year_submission.head(10))

Saved 312 predictions to: /Users/michael/Documents/School/Cornell/AppliedMachineLearning/AML-Midterm/part1/submissions/archive/submitted/kmeans_k2_year__kaggle_0.85149.csv
Validation sample-wise F1: 0.8611
Validation macro F1: 0.6937
Test trials changed from K=2 without year: 19
Test trials requiring fallback: 21


,ID,Predicted
0,TR00756,Safety Efficacy
1,TR05267,Safety
2,TR10354,Safety
3,TR01930,Recruitment
4,TR01046,Safety
5,TR01460,Recruitment
6,TR00199,Recruitment
7,TR09959,Recruitment
8,TR01886,Recruitment
9,TR09553,Other


## Reported Kaggle result: adding year to K=2

Michael reported **0.85149** after submitting `submissions/archive/submitted/kmeans_k2_year__kaggle_0.85149.csv`.

| K=2 classifier | Validation sample-wise F1 | Kaggle score |
|---|---:|---:|
| Embeddings only | 0.8568 | **0.87393** |
| Embeddings plus year | **0.8611** | 0.85149 |

Adding year improved validation by approximately 0.00427 but reduced the reported Kaggle score by **0.02244**, or **2.244 F1 points**. Retain K=2 without year as the best observed Kaggle model among the submitted experiments. This year-feature change did not transfer successfully to the scored test trials.

A plausible explanation is that year relationships learned from the selected/enriched labels and earlier trials do not transfer to later trials. The aggregate Kaggle score alone cannot establish that cause. This result applies to year as a feature in the K=2 augmented classifier; the supervised model plus year remains unsubmitted, and year has not been added to clustering. Preserve both models and record this negative result.


# Safety-only adjustment for later years

Test the saved K=2 model **without year as a classifier feature**. Keep the classifier weights, cluster assignments, generated labels, threshold, and fallback unchanged. Modify only Safety probability after prediction.

Use the same 545-row training dataset to define mean year and standard deviation. For each validation trial, calculate `late_factor = clip((year - training_mean) / training_std, 0, 1)`. A trial at or before the average training year receives no boost; a trial one standard deviation later or more receives the full boost. Compare three fixed maximum probability increases: **0, 0.05, 0.10**. Add `maximum_boost * late_factor` to Safety probability, capped at 1. Other reason probabilities remain unchanged; the shared fallback can still change the selected reason for an otherwise empty prediction set.

The settings are chosen before examining adjusted validation results. This is a heuristic prior adjustment, not calibrated probability estimation. Select primarily by sample-wise F1; exact ties favor no adjustment, then smaller boosts. Also record Safety precision, recall, and false positives. The original model already recovers all 9 Safety-positive validation trials, so a boost cannot recover a missed Safety case in this validation set.

In [21]:
# Load frozen inputs; do not refit the classifier or regenerate labels.
import hashlib
import json
import joblib

SAFETY_MAX_BOOSTS = [0.0, 0.05, 0.10]
SAFETY_REPORTED_KAGGLE_SCORES = {"boost_000": 0.87393, "boost_005": 0.86111}
SAFETY_SCORE_ASSIGNMENT_NOTE = 'Score reported by Michael; assigned to +0.05 because it was the recommended first submission. Boost strength was not explicitly confirmed.'
def resolve_part1_input(directory, name):
    """Locate supplied CSV inputs while leaving experiment outputs under part1/."""
    supplied = {"part1_train.csv", "part1_val.csv", "part1_test.csv", "part1_sample_submission.csv", "pathway_catalog.csv"}
    nested = directory / "data" / name
    return nested if name in supplied and nested.is_file() else directory / name

safety_cwd = Path.cwd()
safety_data_dir = next(
    q for q in [safety_cwd, safety_cwd / "part1", safety_cwd / "AML-Midterm" / "part1"]
    if ((q / "part1_train.csv").is_file() or (q / "data" / "part1_train.csv").is_file())
).resolve()
safety_source_dir = safety_data_dir / "experiments" / "kmeans"
SAFETY_EXPERIMENT_DIR = safety_data_dir / "experiments" / "safety_year_bias"
(SAFETY_EXPERIMENT_DIR / "validation_predictions").mkdir(parents=True, exist_ok=True)
safety_packet_path = safety_source_dir / "models" / "k02.joblib"
safety_packet = joblib.load(safety_packet_path)
safety_model = safety_packet["classifier"]
safety_training_path = safety_source_dir / "datasets" / "k02_train.csv"
safety_training = pd.read_csv(safety_training_path)
safety_val_df = pd.read_csv(resolve_part1_input(safety_data_dir, "part1_val.csv"))
safety_actual = safety_val_df[REASONS].to_numpy(dtype=int)
safety_index = REASONS.index("Safety")
safety_other_indices = [i for i in range(len(REASONS)) if i != safety_index]
assert len(safety_training) == 545 and len(safety_val_df) == 78
assert safety_packet["embedding_columns"] == EMBEDDING_COLUMNS
assert set(safety_training.ID).isdisjoint(safety_val_df.ID)
assert safety_model.named_steps["scaler"].n_features_in_ == 150
safety_year_mean = float(safety_training.start_year.mean())
safety_year_std = float(safety_training.start_year.std(ddof=0))
assert safety_year_std > 0
safety_base_probabilities = safety_model.predict_proba(safety_val_df[EMBEDDING_COLUMNS].to_numpy())
safety_control = evaluate_reason_probabilities(safety_base_probabilities, safety_actual)
safety_recorded = pd.read_csv(safety_source_dir / "comparison.csv").set_index("experiment").loc["k02"]
np.testing.assert_allclose(safety_control["metrics"]["sample_f1"], safety_recorded.validation_sample_f1, atol=1e-14)
np.testing.assert_allclose(safety_control["metrics"]["macro_f1"], safety_recorded.validation_macro_f1, atol=1e-14)
safety_late_factor = np.clip((safety_val_df.start_year.to_numpy() - safety_year_mean) / safety_year_std, 0, 1)
safety_frozen_paths = [
    safety_packet_path, safety_training_path,
    safety_source_dir / "label_audits" / "k02_labels.csv",
    safety_source_dir / "clusters" / "k02_clusters.csv",
    safety_data_dir / "submissions/archive/submitted/logistic_baseline__kaggle_0.87072.csv",
    safety_data_dir / "submissions/archive/submitted/kmeans_k2__kaggle_0.87393.csv",
    safety_data_dir / "submissions/archive/submitted/kmeans_k2_year__kaggle_0.85149.csv",
]
safety_frozen_hashes = {str(q.relative_to(safety_data_dir)): hashlib.sha256(q.read_bytes()).hexdigest()
                        for q in safety_frozen_paths}


def safety_late_year_adjustment(probabilities, years, max_boost):
    """Apply a bounded Safety-only increase using fixed training-year statistics."""
    if max_boost < 0 or max_boost > 1:
        raise ValueError("Maximum probability boost must be between 0 and 1.")
    late_factor = np.clip((np.asarray(years) - safety_year_mean) / safety_year_std, 0, 1)
    adjusted = np.asarray(probabilities).copy()
    adjusted[:, safety_index] = np.minimum(1.0, adjusted[:, safety_index] + max_boost * late_factor)
    return adjusted


def trial_reason_f1(actual, predicted):
    overlap = (actual * predicted).sum(axis=1)
    total = actual.sum(axis=1) + predicted.sum(axis=1)
    return np.divide(2.0 * overlap, total, out=np.zeros(len(actual), dtype=float), where=total != 0)


print(f"Reference year: mean={safety_year_mean:.4f}, standard deviation={safety_year_std:.4f}")
print(f"Full boost begins at year {safety_year_mean + safety_year_std:.4f}.")
print("Fixed maximum boosts:", SAFETY_MAX_BOOSTS)

Reference year: mean=2005.0147, standard deviation=3.2768
Full boost begins at year 2008.2915.
Fixed maximum boosts: [0.0, 0.05, 0.1]


In [22]:
safety_rows = []
safety_evaluations = {}
safety_changed_rows = []
safety_control_trial_f1 = trial_reason_f1(safety_actual, safety_control["predictions"])

for max_boost in SAFETY_MAX_BOOSTS:
    name = f"boost_{round(max_boost * 100):03d}"
    adjusted = safety_late_year_adjustment(safety_base_probabilities, safety_val_df.start_year.to_numpy(), max_boost)
    np.testing.assert_array_equal(adjusted[:, safety_other_indices], safety_base_probabilities[:, safety_other_indices])
    np.testing.assert_array_equal(adjusted[safety_late_factor == 0], safety_base_probabilities[safety_late_factor == 0])
    difference = adjusted[:, safety_index] - safety_base_probabilities[:, safety_index]
    assert (difference >= 0).all() and (difference <= max_boost + 1e-12).all()
    evaluation = evaluate_reason_probabilities(adjusted, safety_actual)
    if max_boost == 0:
        np.testing.assert_array_equal(evaluation["predictions"], safety_control["predictions"])
    predicted = evaluation["predictions"]
    actual_safety = safety_actual[:, safety_index]
    predicted_safety = predicted[:, safety_index]
    tp = int(((actual_safety == 1) & (predicted_safety == 1)).sum())
    fp = int(((actual_safety == 0) & (predicted_safety == 1)).sum())
    fn = int(((actual_safety == 1) & (predicted_safety == 0)).sum())
    changed = (predicted != safety_control["predictions"]).any(axis=1)
    safety_rows.append({
        "experiment": name, "maximum_probability_boost": max_boost,
        "validation_sample_f1": evaluation["metrics"]["sample_f1"],
        "validation_macro_f1": evaluation["metrics"]["macro_f1"],
        "delta_vs_unadjusted_k02": evaluation["metrics"]["sample_f1"] - safety_control["metrics"]["sample_f1"],
        "Safety_f1": evaluation["metrics"]["f1_Safety"],
        "Safety_precision": tp / (tp + fp) if tp + fp else 0.0,
        "Safety_recall": tp / (tp + fn) if tp + fn else 0.0,
        "Safety_true_positives": tp, "Safety_false_positives": fp, "Safety_false_negatives": fn,
        "changed_trials": int(changed.sum()),
        "Safety_prediction_changes": int((predicted_safety != safety_control["predictions"][:, safety_index]).sum()),
        "other_reason_prediction_changes_from_fallback": int((predicted[:, safety_other_indices] != safety_control["predictions"][:, safety_other_indices]).sum()),
        "validation_fallback_trials": evaluation["metrics"]["fallback_trials"],
        **{f"validation_f1_{r}": evaluation["metrics"][f"f1_{r}"] for r in REASONS},
        "kaggle_score": SAFETY_REPORTED_KAGGLE_SCORES.get(name, np.nan),
    })
    output = safety_val_df[["ID", "start_year"] + REASONS].rename(columns={r: f"actual_{r}" for r in REASONS}).copy()
    output["late_year_factor"] = safety_late_factor
    output["Safety_probability_increase"] = difference
    for i, reason in enumerate(REASONS):
        output[f"original_probability_{reason}"] = safety_base_probabilities[:, i]
        output[f"adjusted_probability_{reason}"] = adjusted[:, i]
        output[f"predicted_{reason}"] = predicted[:, i]
    output["used_fallback"] = evaluation["fallback"]
    output["changed_from_unadjusted"] = changed
    output.to_csv(SAFETY_EXPERIMENT_DIR / "validation_predictions" / f"{name}.csv", index=False)
    trial_f1 = trial_reason_f1(safety_actual, predicted)
    for i in np.flatnonzero(changed):
        safety_changed_rows.append({
            "experiment": name, "ID": safety_val_df.iloc[i]["ID"],
            "start_year": int(safety_val_df.iloc[i]["start_year"]),
            "actual_reasons": " ".join(r for r, yes in zip(REASONS, safety_actual[i]) if yes),
            "original_prediction": " ".join(r for r, yes in zip(REASONS, safety_control["predictions"][i]) if yes),
            "adjusted_prediction": " ".join(r for r, yes in zip(REASONS, predicted[i]) if yes),
            "original_Safety_probability": float(safety_base_probabilities[i, safety_index]),
            "adjusted_Safety_probability": float(adjusted[i, safety_index]),
            "original_trial_f1": float(safety_control_trial_f1[i]),
            "adjusted_trial_f1": float(trial_f1[i]),
            "trial_f1_change": float(trial_f1[i] - safety_control_trial_f1[i]),
        })
    safety_evaluations[max_boost] = evaluation

safety_comparison = pd.DataFrame(safety_rows)
safety_comparison.to_csv(SAFETY_EXPERIMENT_DIR / "comparison.csv", index=False)
safety_changed = pd.DataFrame(safety_changed_rows, columns=[
    "experiment", "ID", "start_year", "actual_reasons", "original_prediction", "adjusted_prediction",
    "original_Safety_probability", "adjusted_Safety_probability", "original_trial_f1", "adjusted_trial_f1", "trial_f1_change",
])
safety_changed.to_csv(SAFETY_EXPERIMENT_DIR / "changed_trials.csv", index=False)
safety_selected = safety_comparison.sort_values(
    ["validation_sample_f1", "maximum_probability_boost"], ascending=[False, True], kind="stable"
).iloc[0]
display(safety_comparison[[
    "maximum_probability_boost", "validation_sample_f1", "validation_macro_f1", "Safety_f1",
    "Safety_true_positives", "Safety_false_positives", "Safety_false_negatives", "changed_trials",
]].round(4))
print(f"Selected maximum boost by validation sample-wise F1: {safety_selected.maximum_probability_boost:.2f}")
if len(safety_changed):
    display(safety_changed)
else:
    print("No validation label predictions changed at either nonzero boost.")

,maximum_probability_boost,validation_sample_f1,validation_macro_f1,Safety_f1,Safety_true_positives,Safety_false_positives,Safety_false_negatives,changed_trials
0,0.00,0.8568,0.6808,0.8571,9,3,0,0
1,0.05,0.8440,0.6547,0.8182,9,4,0,1
2,0.10,0.8440,0.6547,0.8182,9,4,0,1


Selected maximum boost by validation sample-wise F1: 0.00


,experiment,ID,start_year,actual_reasons,original_prediction,adjusted_prediction,original_Safety_probability,adjusted_Safety_probability,original_trial_f1,adjusted_trial_f1,trial_f1_change
0,boost_005,TR08868,2008,Other,Other,Safety,0.000087,0.045639,1.0,0.0,-1.0
1,boost_010,TR08868,2008,Other,Other,Safety,0.000087,0.091192,1.0,0.0,-1.0


In [23]:
for name, expected in safety_frozen_hashes.items():
    assert hashlib.sha256((safety_data_dir / name).read_bytes()).hexdigest() == expected
safety_settings = {
    "method": "bounded additive Safety-only probability adjustment for later years",
    "maximum_boost_grid": SAFETY_MAX_BOOSTS,
    "formula": "Safety' = min(1, Safety + max_boost * clip((year - training_mean) / training_std, 0, 1))",
    "year_reference_dataset": "experiments/kmeans/datasets/k02_train.csv",
    "training_year_mean": safety_year_mean, "training_year_std_ddof0": safety_year_std,
    "year_reference_rows": len(safety_training),
    "base_model": "experiments/kmeans/models/k02.joblib", "model_refitted": False,
    "features": EMBEDDING_COLUMNS, "classifier_C": 1000.0,
    "threshold": 0.5, "fallback": "highest-probability reason if no label reaches threshold",
    "non_Safety_probabilities_changed": False,
    "validation_trials": len(safety_val_df), "validation_Safety_positives": int(safety_actual[:, safety_index].sum()),
    "selected_maximum_boost": float(safety_selected.maximum_probability_boost),
    "selection_metric": "validation sample-wise F1; ties favor smaller boost, including zero",
    "frozen_input_sha256": safety_frozen_hashes,
    "validation_sha256": hashlib.sha256((resolve_part1_input(safety_data_dir, "part1_val.csv")).read_bytes()).hexdigest(),
    "kaggle_score_unadjusted": 0.87393,
    "kaggle_score_adjusted": SAFETY_REPORTED_KAGGLE_SCORES["boost_005"],
    "score_assignment_note": SAFETY_SCORE_ASSIGNMENT_NOTE,
    "kaggle_provenance": "Scores reported by Michael; 0.86111 provisionally assigned to +0.05, +0.10 unknown",
}
(SAFETY_EXPERIMENT_DIR / "settings.json").write_text(json.dumps(safety_settings, indent=2) + "\n")

safety_lines = [
    "# Safety-only adjustment for later years", "",
    "Use the frozen K=2 classifier without year as an input feature. Reuse the same 78 validation trials, "
    "threshold 0.5, and nonempty-label fallback. No classifier or cluster model is refitted and no labels or weights change.", "",
    "| Maximum Safety probability increase | Sample-wise F1 | Macro F1 | Safety F1 | Safety TP | Safety FP | Safety FN | Changed trials |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]
for row in safety_rows:
    safety_lines.append(
        f"| {row['maximum_probability_boost']:.2f} | {row['validation_sample_f1']:.4f} | "
        f"{row['validation_macro_f1']:.4f} | {row['Safety_f1']:.4f} | "
        f"{row['Safety_true_positives']} | {row['Safety_false_positives']} | {row['Safety_false_negatives']} | {row['changed_trials']} |"
    )
safety_lines.extend([
    "", "## Method", "",
    "`late_factor = clip((start_year - training_mean_year) / training_std_year, 0, 1)`", "",
    "`adjusted_Safety = min(1, original_Safety + maximum_boost * late_factor)`", "",
    f"Reference statistics come from the existing 545-row K=2 training dataset: mean year **{safety_year_mean:.4f}**, "
    f"standard deviation **{safety_year_std:.4f}**. Full boost starts at year **{safety_year_mean + safety_year_std:.4f}**. "
    "The fixed grid is 0, 0.05, and 0.10: the latter two are capped increases of 5 and 10 percentage points in Safety probability. "
    "These are heuristic adjustments, not calibrated probabilities. Other reason probabilities are left exactly unchanged. "
    "The existing fallback can change an otherwise empty trial's chosen reason when Safety becomes more probable.",
    "", "## Results", "",
    f"Selection by validation sample-wise F1 chooses maximum boost **{safety_selected.maximum_probability_boost:.2f}**, "
    f"with F1 **{safety_selected.validation_sample_f1:.4f}**. Ties favor no adjustment, then the smaller boost.",
    "", "The original K=2 model already identifies all 9 Safety-positive validation trials, with 3 false positives. "
    "There are no missed Safety cases for this adjustment to recover on validation. Added positive Safety predictions "
    "therefore cannot improve Safety recall in this set. This limits what this small validation sample can tell us about "
    "a later-year prior adjustment; it does not establish performance on the later test distribution.",
])
if int(safety_comparison.changed_trials.sum()) == 0:
    safety_lines.extend(["", "Neither nonzero boost changes a validation label prediction. The probabilities changed, "
                         "but not enough to cross a decision threshold or change a fallback choice. The grid therefore "
                         "provides no validation evidence of improvement; retain the unadjusted model."])
else:
    safety_lines.extend(["", "Inspect `changed_trials.csv` for exactly which predictions improved or worsened. "
                         "A larger Safety probability by itself is not a success; the primary criterion is overall sample-wise F1."])
if len(safety_changed):
    safety_lines.extend([
        "", "## Changed predictions", "",
        "| Adjustment | Trial | Year | Actual | Original | Adjusted | Safety probability before | Safety probability after |",
        "|---|---|---:|---|---|---|---:|---:|",
    ])
    for changed in safety_changed.itertuples(index=False):
        safety_lines.append(
            f"| {changed.experiment} | {changed.ID} | {changed.start_year} | {changed.actual_reasons} | "
            f"{changed.original_prediction} | {changed.adjusted_prediction} | "
            f"{changed.original_Safety_probability:.6f} | {changed.adjusted_Safety_probability:.6f} |"
        )
    safety_lines.extend([
        "", "In this run, both boosts change the same trial from a correct Other prediction to an incorrect "
        "Safety prediction. The adjusted Safety probabilities remain below 0.5; the change comes through the "
        "existing highest-probability fallback. This illustrates that probability adjustments can affect "
        "fallback choices even without crossing the classification threshold.",
    ])
safety_lines.extend([
    "", "## Artifacts and controls", "",
    "`comparison.csv` records overall and per-reason F1, Safety precision/recall, error counts, fallback use, "
    "and prediction changes. `validation_predictions/` records raw and adjusted probabilities, year factors, "
    "and labels for all three settings. `changed_trials.csv` records each changed prediction and its trial-wise F1 effect. "
    "`settings.json` records the exact adjustment and checksums of the reused training data and model.",
    "", "Checks confirm the zero-boost control reproduces K=2's recorded validation scores, other reason probabilities "
    "are unchanged, early trials receive no boost, increases stay within their caps, and frozen model/data/submission files "
    "remain unchanged. No new training dataset is created because this experiment only changes postprocessing.",
    "", "The unadjusted K=2 Kaggle score is **0.87393**, reported after manual submission. "
    "A score of 0.86111 was reported for a Safety adjustment, provisionally assigned to the recommended +0.05 file; +0.10 remains unknown. "
    "The earlier full year-feature classifier scored 0.85149; that was a separate intervention.",
])
safety_summary = "\n".join(safety_lines) + "\n"
(SAFETY_EXPERIMENT_DIR / "SUMMARY.md").write_text(safety_summary)
print(safety_summary)
print("Artifacts saved under:", SAFETY_EXPERIMENT_DIR)

# Safety-only adjustment for later years

Use the frozen K=2 classifier without year as an input feature. Reuse the same 78 validation trials, threshold 0.5, and nonempty-label fallback. No classifier or cluster model is refitted and no labels or weights change.

| Maximum Safety probability increase | Sample-wise F1 | Macro F1 | Safety F1 | Safety TP | Safety FP | Safety FN | Changed trials |
|---|---:|---:|---:|---:|---:|---:|---:|
| 0.00 | 0.8568 | 0.6808 | 0.8571 | 9 | 3 | 0 | 0 |
| 0.05 | 0.8440 | 0.6547 | 0.8182 | 9 | 4 | 0 | 1 |
| 0.10 | 0.8440 | 0.6547 | 0.8182 | 9 | 4 | 0 | 1 |

## Method

`late_factor = clip((start_year - training_mean_year) / training_std_year, 0, 1)`

`adjusted_Safety = min(1, original_Safety + maximum_boost * late_factor)`

Reference statistics come from the existing 545-row K=2 training dataset: mean year **2005.0147**, standard deviation **3.2768**. Full boost starts at year **2008.2915**. The fixed grid is 0, 0.05, and 0.10: the latter two are capped in

## Export Safety-adjusted Kaggle submissions

At Michael's request, export both tested nonzero adjustments to check whether their performance transfers differently to Kaggle. Validation selected no adjustment; both nonzero variants scored 0.8440 compared with 0.8568 for the unadjusted K=2 model.

Use exactly the previously tested additive rules with maximum probability increases 0.05 and 0.10, the training-derived year reference, the unchanged saved K=2 classifier, threshold 0.5, and the same fallback. Verify that zero adjustment reproduces the original K=2 submission, then save separate CSVs in sample ID order. Do not fit on test data or change any settings. Compare returned Kaggle scores with **0.87393**. Start with the smaller boost if checking only one variant; if both files contain identical predictions, only one needs uploading.

In [24]:
safety_test_df = pd.read_csv(resolve_part1_input(safety_data_dir, "part1_test.csv"))
safety_sample = pd.read_csv(resolve_part1_input(safety_data_dir, "part1_sample_submission.csv"))
assert safety_test_df.ID.is_unique and safety_sample.ID.is_unique
assert set(safety_test_df.ID) == set(safety_sample.ID)
assert set(safety_test_df.ID).isdisjoint(safety_training.ID)
assert set(safety_test_df.ID).isdisjoint(safety_val_df.ID)
assert np.isfinite(safety_test_df[EMBEDDING_COLUMNS + ["start_year"]].to_numpy()).all()
safety_test_probabilities = safety_model.predict_proba(safety_test_df[EMBEDDING_COLUMNS].to_numpy())


def safety_submission_frame(probabilities):
    predictions, used_fallback = predict_reason_labels(probabilities, threshold=0.5, ensure_nonempty=True)
    frame = pd.DataFrame({
        "ID": safety_test_df.ID,
        "Predicted": [" ".join(r for r, yes in zip(REASONS, row) if yes) for row in predictions],
    }).set_index("ID").loc[safety_sample.ID].reset_index()
    assert frame.columns.tolist() == ["ID", "Predicted"]
    assert frame.shape == (312, 2)
    assert frame.ID.tolist() == safety_sample.ID.tolist()
    assert frame.Predicted.str.len().gt(0).all()
    assert frame.Predicted.map(lambda s: set(s.split()).issubset(REASONS)).all()
    return frame, predictions, used_fallback


safety_unadjusted_frame, safety_test_control, _ = safety_submission_frame(safety_test_probabilities)
safety_original_submission = pd.read_csv(safety_data_dir / "submissions/archive/submitted/kmeans_k2__kaggle_0.87393.csv")
pd.testing.assert_frame_equal(safety_unadjusted_frame, safety_original_submission)
safety_submission_rows = []
safety_submission_predictions = {}
for max_boost in [boost for boost in SAFETY_MAX_BOOSTS if boost > 0]:
    adjusted = safety_late_year_adjustment(safety_test_probabilities, safety_test_df.start_year.to_numpy(), max_boost)
    np.testing.assert_array_equal(adjusted[:, safety_other_indices], safety_test_probabilities[:, safety_other_indices])
    frame, predicted, fallback = safety_submission_frame(adjusted)
    boost_name = f"boost_{round(max_boost * 100):03d}"
    path = safety_data_dir / {"boost_005": "submissions/archive/submitted/kmeans_k2_safety_boost005__reported_0.86111.csv", "boost_010": "submissions/archive/unsubmitted/kmeans_k2_safety_boost010__157labels.csv"}[boost_name]
    frame.to_csv(path, index=False)
    pd.testing.assert_frame_equal(pd.read_csv(path), frame)
    validation = safety_comparison.set_index("experiment").loc[boost_name]
    safety_submission_rows.append({
        "experiment": boost_name, "maximum_probability_boost": max_boost,
        "submission_file": str(path.relative_to(safety_data_dir)), "test_rows": len(frame),
        "test_trials_changed_from_unadjusted": int((predicted != safety_test_control).any(axis=1).sum()),
        "test_Safety_predictions": int(predicted[:, safety_index].sum()),
        "test_fallback_trials": int(fallback.sum()),
        "validation_sample_f1": float(validation.validation_sample_f1),
        "validation_macro_f1": float(validation.validation_macro_f1),
        "kaggle_score": SAFETY_REPORTED_KAGGLE_SCORES.get(boost_name),
        "kaggle_status": "reported; assumed +0.05" if boost_name == "boost_005" else "pending manual submission",
        "submission_sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    })
    safety_submission_predictions[max_boost] = predicted
safety_submissions_identical = np.array_equal(safety_submission_predictions[0.05], safety_submission_predictions[0.10])
safety_submissions = pd.DataFrame(safety_submission_rows)
safety_submissions.to_csv(SAFETY_EXPERIMENT_DIR / "submissions.csv", index=False)
safety_export_record = {
    "submissions": safety_submission_rows,
    "prediction_files_identical": bool(safety_submissions_identical),
    "kaggle_comparator_unadjusted": 0.87393,
    "model_file": "experiments/kmeans/models/k02.joblib",
    "model_sha256": hashlib.sha256(safety_packet_path.read_bytes()).hexdigest(),
    "training_year_mean": safety_year_mean, "training_year_std": safety_year_std,
    "formula": safety_settings["formula"], "threshold": 0.5,
    "fallback": safety_settings["fallback"], "validation_selected_maximum_boost": float(safety_selected.maximum_probability_boost),
    "purpose": "User-requested Kaggle comparison of previously tested Safety adjustments",
    "score_assignment_note": SAFETY_SCORE_ASSIGNMENT_NOTE,
}
(SAFETY_EXPERIMENT_DIR / "submission_manifest.json").write_text(json.dumps(safety_export_record, indent=2) + "\n")
safety_export_summary_path = SAFETY_EXPERIMENT_DIR / "SUMMARY.md"
safety_export_summary = safety_export_summary_path.read_text().split("## Safety-adjusted Kaggle exports")[0].rstrip()
safety_export_lines = [
    "", "", "## Safety-adjusted Kaggle exports", "",
    "The following files were prepared at Michael's request to compare with unadjusted K=2 at **0.87393**. "
    "Both nonzero variants scored 0.8440 on validation; no adjustment remains the validation-selected rule.", "",
    "| Maximum boost | File | Changed test trials | Safety predictions | Kaggle status |",
    "|---|---|---:|---:|---|",
]
for row in safety_submission_rows:
    safety_export_lines.append(
        f"| {row['maximum_probability_boost']:.2f} | `{row['submission_file']}` | "
        f"{row['test_trials_changed_from_unadjusted']} | {row['test_Safety_predictions']} | {row['kaggle_score']} ({row['kaggle_status']}) |"
    )
safety_export_lines.extend([
    "", f"The two files contain identical predictions: **{safety_submissions_identical}**. "
    "If identical, upload only one. Otherwise, start with the smaller adjustment when checking only one variant. "
    "`submissions.csv` and `submission_manifest.json` record file hashes and settings.", "",
])
safety_export_lines.append('## Reported Safety-adjustment Kaggle result\n\nMichael reported **0.86111**. This is provisionally assigned to the recommended **+0.05** submission; the boost strength was not explicitly stated. The +0.10 result remains unknown.\n\n| Rule | Validation sample-wise F1 | Kaggle score |\n|---|---:|---:|\n| Unadjusted K=2 | **0.8568** | **0.87393** |\n| Tested Safety adjustment (assumed +0.05) | 0.8440 | 0.86111 |\n\nThe observed Kaggle difference is **-0.01282**, approximately **-1.28 F1 points**. The validation difference is approximately **-0.01282** as well. Validation and Kaggle agree in direction and almost exactly in size for this particular adjustment. Retain unadjusted K=2 as the best observed Kaggle model.\n\nThis agreement does not establish a reliable general correlation: K=2 versus the supervised baseline and the addition of year previously produced opposite validation/Kaggle directions. Use validation as a useful but imperfect guide, with occasional Kaggle checks as agreed. This result argues against this tested Safety boost, not every possible temporal method.\n')
safety_export_summary_path.write_text(safety_export_summary + "\n".join(safety_export_lines))
for name, expected in safety_frozen_hashes.items():
    assert hashlib.sha256((safety_data_dir / name).read_bytes()).hexdigest() == expected
display(safety_submissions.drop(columns=["submission_sha256"]).round(4))
print("Two adjusted submissions identical:", safety_submissions_identical)
print("Unadjusted K=2 submission was reproduced exactly and remains unchanged.")

,experiment,maximum_probability_boost,submission_file,test_rows,test_trials_changed_from_unadjusted,test_Safety_predictions,test_fallback_trials,validation_sample_f1,validation_macro_f1,kaggle_score,kaggle_status
0,boost_005,0.05,submissions/archive/submitted/kmeans_k2_safety_boost005__reported_0.86111.csv,312,7,64,24,0.844,0.6547,None,pending manual submission
1,boost_010,0.10,submissions/archive/unsubmitted/kmeans_k2_safety_boost010__157labels.csv,312,10,67,22,0.844,0.6547,None,pending manual submission


Two adjusted submissions identical: False
Unadjusted K=2 submission was reproduced exactly and remains unchanged.


## Reported Safety-adjustment Kaggle result

Michael reported **0.86111**. This is provisionally assigned to the recommended **+0.05** submission; the boost strength was not explicitly stated. The +0.10 result remains unknown.

| Rule | Validation sample-wise F1 | Kaggle score |
|---|---:|---:|
| Unadjusted K=2 | **0.8568** | **0.87393** |
| Tested Safety adjustment (assumed +0.05) | 0.8440 | 0.86111 |

The observed Kaggle difference is **-0.01282**, approximately **-1.28 F1 points**. The validation difference is approximately **-0.01282** as well. Validation and Kaggle agree in direction and almost exactly in size for this particular adjustment. Retain unadjusted K=2 as the best observed Kaggle model.

This agreement does not establish a reliable general correlation: K=2 versus the supervised baseline and the addition of year previously produced opposite validation/Kaggle directions. Use validation as a useful but imperfect guide, with occasional Kaggle checks as agreed. This result argues against this tested Safety boost, not every possible temporal method.
